# 다음에 올 단어 맞히기 — RNN 과 LSTM (TensorFlow 실습)

**「04. RNN / LSTM」 실습 노트북 — v1.2**

한국어 업무 문장의 **앞부분**을 읽고, 바로 **다음에 올 단어 후보**를 제안하는 작은 신경망을 만듭니다.

| 입력 — 문장의 앞부분 | 모델 | 출력 — 다음 단어 후보 |
|---|---|---|
| 자료를 확인하고 담당자에게 | SimpleRNN / LSTM | 전달했습니다 … |
| 회의 자료를 검토한 후 결과를 | SimpleRNN / LSTM | 공유했습니다 … |

---

## 이 노트북에서 이해할 단 하나

> ### 모델은 지금 들어온 단어만 보는 것이 아닙니다.
> ### **이전 시점의 상태(Hidden State)를 다음 시점으로 계속 전달합니다.**

이 한 가지를 이해하는 것이 이번 실습의 전부입니다.

---

## 전체 흐름

| 순서 | 단계 | 하는 일 |
|---|---|---|
| 1 | 문장 | 교육용 업무 문장 |
| 2 | **단어 토큰** | 문장을 띄어쓰기 기준으로 나눔 |
| 3 | **Token ID** | 토큰마다 번호를 붙임 |
| 4 | **Embedding** | 번호를 학습 가능한 숫자 벡터로 바꿈 |
| 5 | **SimpleRNN / LSTM** | ★ 순서대로 읽으며 **상태를 다음 시점으로 전달** |
| 6 | **Dense + Softmax** | 다음 단어 후보 점수 계산 |
| 7 | 결과 | 다음 단어 후보 |

---

## 이 노트북이 다루는 것과 다루지 않는 것

| | 다루는 것 | 다루지 않는 것 |
|---|---|---|
| 목적 | RNN / LSTM 의 **상태 전달** 이해 | 모델 성능 올리기 |
| 코드 | 꼭 필요한 13개 셀 | 그래프 · 통계 · 평가 지표 |
| 결과 | 다음 단어 후보 3개 | 정확도 벤치마크 |

> **이번 실습은 RNN / LSTM 의 동작 원리를 이해하기 위한 교육용 예제이며, 정식 성능평가 실습은 아닙니다.**
> 문장 수가 매우 적기 때문에 정확도 수치에는 의미를 두지 않습니다.

> **안내**
> 사용하는 문장은 **전부 교육용으로 새로 지어낸 가상의 업무 문장**입니다.
> 실제 문서 · 내부 자료 · 개인정보는 사용하지 않았고, 외부 데이터도 내려받지 않습니다.

> **실행 방법**
> 위에서부터 `Shift + Enter` 로 순서대로 실행하면 됩니다. 전체 1~2분 정도 걸립니다.

---
## 실행 준비

### 지금 무엇을 하나요?
계산 도구 두 개(`numpy`, `tensorflow`)만 불러오고, 실행할 때마다 같은 결과가 나오도록 난수 씨앗을 고정합니다.

### 왜 이것뿐인가요?
그래프를 그리거나 표를 만드는 도구는 이번 실습에 필요하지 않습니다.
**RNN / LSTM 자체에 집중하기 위해 도구를 최소한으로 줄였습니다.**

In [1]:
import os

# TensorFlow 의 정보성 로그를 줄여 실습 화면을 깔끔하게 유지합니다.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import tensorflow as tf

tf.get_logger().setLevel("ERROR")

# 실행할 때마다 같은 결과가 나오도록 난수 씨앗을 고정합니다.
SEED = 42
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow 버전 :", tf.__version__)
print("NumPy 버전      :", np.__version__)

TensorFlow 버전 : 2.21.0
NumPy 버전      : 2.4.6


---
# 01. 왜 순서가 중요한가

## 01-1. 같은 단어, 다른 순서

두 문장에 들어 있는 낱말은 상당 부분 겹칩니다. 그런데 **일이 일어난 순서가 다릅니다.**

- **순서 A** — 자료를 먼저 확인하고, 그 다음에 전달했습니다.
- **순서 B** — 먼저 전달하고, 자료 확인은 그 뒤였습니다.


<svg viewBox="0 0 940 520" width="100%" role="img" aria-label="같은 단어로 이루어졌지만 처리 순서가 다른 두 문장의 비교" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>같은 단어, 다른 순서</title><desc>왼쪽은 자료를 확인하고 담당자에게 전달했습니다 순서로, 오른쪽은 담당자에게 전달하고 자료를 확인했습니다 순서로 단어를 위에서 아래로 처리하는 흐름을 나란히 그려, 같은 단어가 쓰여도 처리 순서가 다르면 사건의 흐름이 달라짐을 보여 주는 그림</desc><rect x="0" y="0" width="940" height="520" fill="#FFFFFF"/><text x="470.0" y="34.0" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">같은 단어가 쓰여도 순서가 다르면 흐름이 달라집니다</text><text x="470.0" y="58.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">두 문장에 들어 있는 단어는 상당 부분 겹치지만, 읽는 순서가 다릅니다</text><rect x="70.0" y="80.0" width="360.0" height="350.0" rx="10" ry="10" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="250.0" y="108.0" font-size="17" fill="#1B3A5E" text-anchor="middle" font-weight="bold">순서 A</text><text x="250.0" y="128.0" font-size="12.5" fill="#6B7280" text-anchor="middle" font-weight="normal">자료 확인이 먼저입니다</text><rect x="130.0" y="142.0" width="240.0" height="44.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="250.0" y="169.4" font-size="15" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자료를</text><line x1="250.0" y1="186.0" x2="250.0" y2="199.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,208.0 256.0,199.0 244.0,199.0" fill="#8A93A0"/><rect x="130.0" y="212.0" width="240.0" height="44.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="250.0" y="239.4" font-size="15" fill="#1B3A5E" text-anchor="middle" font-weight="normal">확인하고</text><line x1="250.0" y1="256.0" x2="250.0" y2="269.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,278.0 256.0,269.0 244.0,269.0" fill="#8A93A0"/><rect x="130.0" y="282.0" width="240.0" height="44.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="250.0" y="309.4" font-size="15" fill="#1B3A5E" text-anchor="middle" font-weight="normal">담당자에게</text><line x1="250.0" y1="326.0" x2="250.0" y2="339.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,348.0 256.0,339.0 244.0,339.0" fill="#8A93A0"/><rect x="130.0" y="352.0" width="240.0" height="44.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="250.0" y="379.4" font-size="15" fill="#1D4726" text-anchor="middle" font-weight="normal">전달했습니다</text><text x="250.0" y="418.0" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="normal">마지막 단어</text><rect x="510.0" y="80.0" width="360.0" height="350.0" rx="10" ry="10" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="690.0" y="108.0" font-size="17" fill="#0E4A46" text-anchor="middle" font-weight="bold">순서 B</text><text x="690.0" y="128.0" font-size="12.5" fill="#6B7280" text-anchor="middle" font-weight="normal">전달이 먼저입니다</text><rect x="570.0" y="142.0" width="240.0" height="44.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="690.0" y="169.4" font-size="15" fill="#0E4A46" text-anchor="middle" font-weight="normal">담당자에게</text><line x1="690.0" y1="186.0" x2="690.0" y2="199.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,208.0 696.0,199.0 684.0,199.0" fill="#8A93A0"/><rect x="570.0" y="212.0" width="240.0" height="44.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="690.0" y="239.4" font-size="15" fill="#0E4A46" text-anchor="middle" font-weight="normal">전달하고</text><line x1="690.0" y1="256.0" x2="690.0" y2="269.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,278.0 696.0,269.0 684.0,269.0" fill="#8A93A0"/><rect x="570.0" y="282.0" width="240.0" height="44.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="690.0" y="309.4" font-size="15" fill="#0E4A46" text-anchor="middle" font-weight="normal">자료를</text><line x1="690.0" y1="326.0" x2="690.0" y2="339.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,348.0 696.0,339.0 684.0,339.0" fill="#8A93A0"/><rect x="570.0" y="352.0" width="240.0" height="44.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="690.0" y="379.4" font-size="15" fill="#1D4726" text-anchor="middle" font-weight="normal">확인했습니다</text><text x="690.0" y="418.0" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="normal">마지막 단어</text><text x="470.0" y="468.0" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="bold">같은 단어가 포함되어 있어도 처리 순서가 다르면 상태의 흐름도 달라집니다.</text><text x="470.0" y="492.0" font-size="13.0" fill="#6B7280" text-anchor="middle" font-weight="normal">그래서 다음 단어를 예측하려면 단어를 나온 순서 그대로 읽는 구조가 필요합니다.</text></svg>


### 여기서 기억할 것

> **사용된 단어가 비슷해도 순서가 다르면 사건의 흐름이 달라집니다.**

---
## 01-2. 평균 방식과 순차 방식

앞에서 실행한 **02. 딥러닝 민원 분류 실습**과 이번 실습은 문장을 다루는 방법이 다릅니다.

| | 앞의 딥러닝 실습 (02번) | 이번 RNN / LSTM 실습 |
|---|---|---|
| 나누는 단위 | **글자 토큰** | **띄어쓰기 기준 단어 토큰** |
| 처리 방법 | 글자 토큰의 Embedding 벡터를 **평균**내어 문장 전체를 하나의 벡터로 요약 | 단어 토큰을 **순서대로 하나씩** 처리 |
| 순서 정보 | 여러 위치의 표현을 하나로 합침 | 읽은 순서대로 상태를 갱신 |
| 묻는 질문 | 이 문장은 어떤 종류인가? | 다음에 어떤 단어가 올까? |

### 평균 방식

> **평균은 여러 위치의 표현을 하나로 합치기 때문에 순서 정보가 직접 유지되지 않습니다.**

“모든 정보가 사라진다”는 뜻은 아닙니다. **위치와 순서가 최종 요약 벡터에 직접 남지 않는다**는 뜻입니다.

### 순차 방식 (이번 실습)

> **RNN 은 단어 토큰을 읽은 순서대로 상태를 갱신합니다.**

단어 토큰 1을 읽어 상태를 만들고, 그 상태를 가지고 단어 토큰 2를 읽어 새 상태를 만들고,
다시 그 상태를 가지고 단어 토큰 3을 읽습니다. 이 과정을 05장에서 그림으로 확인합니다.

---
# 02. 교육용 업무 문장

## 02-1. 기본 업무 문장

### 지금 무엇을 하나요?
모델에게 읽힐 **짧은 업무 문장 28개**를 직접 적어 둡니다.

### 왜 이렇게 적나요?
문장을 자동으로 만들어 내는 복잡한 코드는 RNN / LSTM 과 아무 관계가 없습니다.
그래서 **눈으로 바로 읽을 수 있게 문장을 그대로 적었습니다.**

### 문장을 고른 기준

앞부분에 따라 **뒤에 오는 단어가 달라지도록** 구성했습니다.
모델이 “무슨 문장이든 항상 같은 단어”만 답하지 않도록 하기 위해서입니다.

| 앞부분이 이렇게 끝나면 | 다음 단어는 |
|---|---|
| … 담당자에게 | **전달했습니다** |
| … 결과를 | **공유했습니다** |
| … 민원인에게 | **안내했습니다** |
| … 서류를 | **제출했습니다** |
| … 내용을 | **확인했습니다** |
| … 담당 부서에 | **보고했습니다** |
| … 추가 자료를 | **요청했습니다** |
| … 회의 내용을 | **정리했습니다** |

In [2]:
# 교육용 가상 업무 문장 (외부 데이터를 사용하지 않습니다)
corpus = """
자료를 확인하고 담당자에게 전달했습니다
관련 서류를 검토한 후 담당자에게 전달했습니다
요청된 문서를 준비하여 담당자에게 전달했습니다
접수 자료를 정리하여 담당자에게 전달했습니다
회의 자료를 검토한 후 결과를 공유했습니다
회의 내용을 정리한 뒤 결과를 공유했습니다
논의 사항을 확인한 후 결과를 공유했습니다
협의 내용을 검토한 뒤 결과를 공유했습니다
신청서를 접수한 후 민원인에게 안내했습니다
민원 내용을 검토한 뒤 민원인에게 안내했습니다
처리 결과를 확인한 후 민원인에게 안내했습니다
문의 내용을 정리한 뒤 민원인에게 안내했습니다
보완 요청을 받은 후 서류를 제출했습니다
추가 서류 요청에 따라 서류를 제출했습니다
미비 사항을 보완한 뒤 서류를 제출했습니다
전달받은 자료의 내용을 확인했습니다
회신 문서를 받은 후 내용을 확인했습니다
접수된 서류의 내용을 확인했습니다
진행 상황을 점검한 뒤 담당 부서에 보고했습니다
처리 경과를 확인한 후 담당 부서에 보고했습니다
검토 결과를 정리한 뒤 담당 부서에 보고했습니다
검토에 필요한 자료가 부족하여 추가 자료를 요청했습니다
확인 과정에서 빠진 항목이 있어 추가 자료를 요청했습니다
회의 기록을 검토한 뒤 회의 내용을 정리했습니다
논의 결과를 확인한 후 회의 내용을 정리했습니다
보완 요청을 받은 후 관련 자료를 검토하고 필요한 내용을 정리하여 담당자에게 제출했습니다
지난 회의에서 정한 기준에 따라 관련 자료를 검토하고 필요한 내용을 정리하여 담당자에게 보고했습니다
민원인의 문의를 접수한 후 관련 자료를 검토하고 필요한 내용을 정리하여 담당자에게 안내했습니다
""".strip().splitlines()

print("문장 수 :", len(corpus), "개")
print()
print("가장 짧은 문장 :", min(corpus, key=lambda s: len(s.split())))
print("가장 긴 문장   :", max(corpus, key=lambda s: len(s.split())))

문장 수 : 28 개

가장 짧은 문장 : 자료를 확인하고 담당자에게 전달했습니다
가장 긴 문장   : 지난 회의에서 정한 기준에 따라 관련 자료를 검토하고 필요한 내용을 정리하여 담당자에게 보고했습니다


---
## 02-2. 긴 문맥 예제 — 앞의 정보가 마지막 단어를 결정합니다

맨 아래 세 문장은 **일부러 길게** 만들었습니다.
세 문장의 **가운데 여덟 단어는 완전히 같고**, 맨 앞의 도입 부분만 다릅니다.


<svg viewBox="0 0 940 512" width="100%" role="img" aria-label="문장 앞의 도입 부분이 문장 마지막 단어를 결정하는 세 문장의 구조" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>긴 문맥 의존 구조</title><desc>세 문장 모두 가운데 여덟 단어가 완전히 같고 맨 앞의 도입 부분만 다른데 마지막 단어가 각각 제출했습니다 보고했습니다 안내했습니다로 달라지는 구조를 가로로 나란히 그려, 마지막 단어를 맞히려면 멀리 떨어진 앞부분 정보가 필요함을 보여 주는 그림</desc><rect x="0" y="0" width="940" height="512" fill="#FFFFFF"/><text x="470.0" y="34.0" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">가운데는 똑같고, 맨 앞이 마지막 단어를 결정합니다</text><text x="470.0" y="58.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">세 문장의 가운데 여덟 단어는 완전히 같습니다</text><text x="470.0" y="80.0" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">점선 화살표 = 맨 앞의 정보가 마지막 단어를 결정합니다</text><rect x="40.0" y="110.0" width="258.0" height="52.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="169.0" y="140.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">보완 요청을 받은 후</text><rect x="312.0" y="110.0" width="356.0" height="52.0" rx="10" ry="10" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="490.0" y="140.3" font-size="12" fill="#555C68" text-anchor="middle" font-weight="normal">관련 자료를 검토하고 필요한 내용을 정리하여 담당자에게</text><rect x="682.0" y="110.0" width="218.0" height="52.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="791.0" y="141.0" font-size="14" fill="#1D4726" text-anchor="middle" font-weight="normal">제출했습니다</text><line x1="298.0" y1="136.0" x2="302.0" y2="136.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="310.0,136.0 302.0,131.0 302.0,141.0" fill="#8A93A0"/><line x1="668.0" y1="136.0" x2="672.0" y2="136.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="680.0,136.0 672.0,131.0 672.0,141.0" fill="#8A93A0"/><path d="M 169 164 C 169 196 791 196 791 170" fill="none" stroke="#4C78A8" stroke-width="2.0" stroke-dasharray="5 4"/><line x1="791.0" y1="182.0" x2="791.0" y2="176.0" stroke="#4C78A8" stroke-width="2.0"/><polygon points="791.0,168.0 786.0,176.0 796.0,176.0" fill="#4C78A8"/><rect x="40.0" y="228.0" width="258.0" height="52.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="169.0" y="258.9" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">지난 회의에서 정한 기준에 따라</text><rect x="312.0" y="228.0" width="356.0" height="52.0" rx="10" ry="10" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="490.0" y="258.3" font-size="12" fill="#555C68" text-anchor="middle" font-weight="normal">관련 자료를 검토하고 필요한 내용을 정리하여 담당자에게</text><rect x="682.0" y="228.0" width="218.0" height="52.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="791.0" y="259.0" font-size="14" fill="#1D4726" text-anchor="middle" font-weight="normal">보고했습니다</text><line x1="298.0" y1="254.0" x2="302.0" y2="254.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="310.0,254.0 302.0,249.0 302.0,259.0" fill="#8A93A0"/><line x1="668.0" y1="254.0" x2="672.0" y2="254.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="680.0,254.0 672.0,249.0 672.0,259.0" fill="#8A93A0"/><path d="M 169 282 C 169 314 791 314 791 288" fill="none" stroke="#2E9E96" stroke-width="2.0" stroke-dasharray="5 4"/><line x1="791.0" y1="300.0" x2="791.0" y2="294.0" stroke="#2E9E96" stroke-width="2.0"/><polygon points="791.0,286.0 786.0,294.0 796.0,294.0" fill="#2E9E96"/><rect x="40.0" y="346.0" width="258.0" height="52.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="169.0" y="376.9" font-size="13.5" fill="#3B295D" text-anchor="middle" font-weight="normal">민원인의 문의를 접수한 후</text><rect x="312.0" y="346.0" width="356.0" height="52.0" rx="10" ry="10" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="490.0" y="376.3" font-size="12" fill="#555C68" text-anchor="middle" font-weight="normal">관련 자료를 검토하고 필요한 내용을 정리하여 담당자에게</text><rect x="682.0" y="346.0" width="218.0" height="52.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="791.0" y="377.0" font-size="14" fill="#1D4726" text-anchor="middle" font-weight="normal">안내했습니다</text><line x1="298.0" y1="372.0" x2="302.0" y2="372.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="310.0,372.0 302.0,367.0 302.0,377.0" fill="#8A93A0"/><line x1="668.0" y1="372.0" x2="672.0" y2="372.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="680.0,372.0 672.0,367.0 672.0,377.0" fill="#8A93A0"/><path d="M 169 400 C 169 432 791 432 791 406" fill="none" stroke="#7B5EA7" stroke-width="2.0" stroke-dasharray="5 4"/><line x1="791.0" y1="418.0" x2="791.0" y2="412.0" stroke="#7B5EA7" stroke-width="2.0"/><polygon points="791.0,404.0 786.0,412.0 796.0,412.0" fill="#7B5EA7"/><text x="470.0" y="452.0" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="bold">마지막 단어를 제대로 예측하려면 앞부분의 정보가 멀리까지 영향을 줄 수 있어야 합니다.</text><text x="470.0" y="476.0" font-size="13.0" fill="#6B7280" text-anchor="middle" font-weight="normal">바로 앞 단어(담당자에게)만 보아서는 세 문장을 구분할 수 없습니다.</text></svg>


### 여기서 기억할 것

- 바로 앞 단어(`담당자에게`)만 보면 세 문장을 **구분할 수 없습니다.**
- 마지막 단어를 맞히려면 **여덟 단어 앞의 도입 부분을 계속 유지**해야 합니다.
- 이 예제는 **08장에서 LSTM 이 왜 필요한지** 설명할 때 다시 사용합니다.

---
# 03. 텍스트를 단어 토큰으로 준비

## 03-1. 띄어쓰기 기준 토큰

### Token 이란?

> **Token** — 문장을 모델이 처리하기 위해 나눈 작은 단위입니다.

토큰을 어떤 크기로 나눌지는 모델마다 다릅니다. 글자 단위, 어절 단위, 그 중간인 서브워드 단위가 모두 쓰입니다.

### 이번 실습에서 나누는 방법

이번 실습의 코드는 `split="whitespace"` 를 사용합니다. 즉 **띄어쓰기를 기준으로 나눕니다.**

| 원래 문장 | 나눈 결과 |
|---|---|
| 자료를 확인하고 담당자에게 | `자료를` · `확인하고` · `담당자에게` |
| 민원 담당자에게 | `민원` · `담당자에게` |

> **이번 실습에서는 문장을 띄어쓰기 기준으로 나눕니다.
> 이해를 쉽게 하기 위해 이후에는 각각을 ‘단어 토큰’이라고 부르겠습니다.**

한국어에서 띄어쓰기 단위가 언어학적 의미의 ‘단어’와 항상 같지는 않습니다.
`민원 담당자에게` 처럼 띄어쓰기가 들어간 표현은 **두 개의 단어 토큰**이 됩니다.

---
## 03-2. Token ID

### Token ID 란?

> **Token ID** — 각 토큰을 구분하기 위해 붙인 번호입니다.

신경망은 글자를 그대로 읽지 못하므로, 먼저 **단어 토큰마다 번호를 붙입니다.**

이 번호는 **단순한 이름표**입니다. 40번이 4번보다 크다는 뜻은 전혀 없습니다.
번호에 의미를 담는 일은 다음 단계인 `Embedding` 이 합니다.

### 아래 코드가 만드는 것

| 이름 | 뜻 |
|---|---|
| `vectorize` | 문장을 Token ID 배열로 바꿔 주는 도구 |
| `어휘` | 번호 ↔ 단어 토큰 대응표 |
| `VOCAB_SIZE` | 사전에 들어 있는 단어 토큰의 개수 |
| `MAX_LEN` | 가장 긴 문장의 토큰 수 (입력 칸의 개수) |

> 이 단계는 **RNN / LSTM 의 핵심이 아닌 준비 단계**입니다.
> `TextVectorization` 의 세부 옵션은 외울 필요가 없습니다.

In [3]:
MAX_LEN = max(len(문장.split()) for 문장 in corpus)   # 가장 긴 문장의 단어 수

vectorize = tf.keras.layers.TextVectorization(
    standardize=None,                  # 한국어 원문을 그대로 둡니다.
    split="whitespace",                # 띄어쓰기 기준으로 단어를 자릅니다.
    output_mode="int",                 # 단어를 번호로 바꿉니다.
    output_sequence_length=MAX_LEN,    # 모든 문장을 같은 길이로 맞춥니다.
)
vectorize.adapt(np.array(corpus, dtype=object))

어휘 = [str(단어) for 단어 in vectorize.get_vocabulary()]   # 번호 → 단어 대응표
VOCAB_SIZE = len(어휘)

print("단어 사전 크기 VOCAB_SIZE :", VOCAB_SIZE, "개")
print("입력 칸 개수    MAX_LEN    :", MAX_LEN, "칸")
print()

예시 = "자료를 확인하고 담당자에게"
번호 = [int(t) for t in vectorize(np.array([예시], dtype=object)).numpy()[0] if t != 0]
print(f'"{예시}"')
print("        ↓")
print("     ", 번호)
print("        ↓  다시 단어로 되돌리면")
print("     ", [어휘[t] for t in 번호])

단어 사전 크기 VOCAB_SIZE : 75 개
입력 칸 개수    MAX_LEN    : 13 칸

"자료를 확인하고 담당자에게"
        ↓
      [4, 40, 6]
        ↓  다시 단어로 되돌리면
      ['자료를', '확인하고', '담당자에게']


---
## 03-3. Embedding

### Embedding 이란?

> **Embedding** — Token ID 하나를 모델이 학습할 수 있는 **숫자 벡터**로 바꾸는 층입니다.

Embedding 은 “단어만을 위한 기능”이 아닙니다. **토큰이면 무엇이든** 벡터로 바꿉니다.
토큰은 모델에 따라 **글자 · 어절/단어 · 서브워드** 등이 될 수 있습니다.

| | 값 | 설명 |
|---|---|---|
| 들어가는 것 | Token ID | 예: `자료를` → 4번 |
| Embedding 층 | `Embedding(VOCAB_SIZE, 32)` | 번호마다 숫자 32개짜리 벡터를 학습 |
| 나오는 것 | 숫자 벡터 | 이 벡터가 RNN / LSTM 에 실제로 들어갑니다 |

> **이번 노트북에서는 띄어쓰기 기준 단어 토큰을 사용합니다.**

### 왜 중요한가요?

RNN 에 실제로 들어가는 것은 `"자료를"` 이라는 **글자 자체가 아니라**,
Embedding 을 거쳐 나온 **숫자 벡터**입니다. 05장의 그림에서 이 벡터를 `x` 로 표시합니다.

---
# 04. 다음 단어 학습 문제

## 04-1. 입력 → 다음 단어 정답

### 정답을 사람이 적어 줄 필요가 없습니다

완성된 문장 하나를 **“지금까지 읽은 토큰 → 바로 다음 토큰”** 형태의 연습 문제로 바꿉니다.
문장 안의 **바로 다음 단어 토큰이 그대로 정답**이 되므로, 문장만 있으면 학습 데이터가 자동으로 만들어집니다.


<svg width="100%" viewBox="0 0 940 560" role="img" aria-label="한 문장이 다음 단어 예측 학습 데이터로 바뀌는 과정" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>한 문장이 다음 단어 예측 학습 데이터로 바뀌는 과정</title><desc>자료를 확인하고 담당자에게 전달했습니다 라는 한 문장에서, 앞에서부터 단어를 하나씩 늘려 가며 그 다음 단어를 정답으로 하는 학습 예제 세 개가 자동으로 만들어지는 과정을 보여 주는 그림</desc><rect x="0" y="0" width="940" height="560" fill="#FFFFFF"/><text x="470.0" y="34" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">문장 하나가 다음 단어 맞히기 문제 여러 개가 됩니다</text><text x="470.0" y="58" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">지금까지 나온 단어를 보고 바로 다음 단어를 맞히는 연습 문제를 자동으로 만듭니다</text><text x="96" y="105" font-size="14" fill="#6B7280" text-anchor="start" font-weight="bold">원문</text><rect x="175" y="84" width="150" height="42" rx="8" ry="8" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="250.0" y="110.0" font-size="14.5" fill="#555C68" text-anchor="middle" font-weight="normal">자료를</text><rect x="335" y="84" width="150" height="42" rx="8" ry="8" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="410.0" y="110.0" font-size="14.5" fill="#555C68" text-anchor="middle" font-weight="normal">확인하고</text><rect x="495" y="84" width="150" height="42" rx="8" ry="8" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="570.0" y="110.0" font-size="14.5" fill="#555C68" text-anchor="middle" font-weight="normal">담당자에게</text><rect x="655" y="84" width="150" height="42" rx="8" ry="8" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="730.0" y="110.0" font-size="14.5" fill="#555C68" text-anchor="middle" font-weight="normal">전달했습니다</text><line x1="96" y1="146" x2="844" y2="146" stroke="#CBD0D8" stroke-width="1.5"/><text x="272" y="178" font-size="13.5" fill="#6B7280" text-anchor="middle" font-weight="bold">입력 — 지금까지 읽은 단어</text><text x="742" y="178" font-size="13.5" fill="#1D4726" text-anchor="middle" font-weight="bold">정답 — 바로 다음 단어</text><circle cx="118" cy="224" r="13" fill="#7B5EA7"/><text x="118" y="229" font-size="13.5" fill="#FFFFFF" text-anchor="middle" font-weight="bold">1</text><rect x="175" y="198" width="150" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="250.0" y="224.0" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자료를</text><line x1="333" y1="219" x2="647" y2="219" stroke="#8A93A0" stroke-width="2"/><polygon points="656,219 646,213 646,225" fill="#8A93A0"/><rect x="666" y="198" width="150" height="42" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="741.0" y="224.0" font-size="14.5" fill="#1D4726" text-anchor="middle" font-weight="bold">확인하고</text><circle cx="118" cy="290" r="13" fill="#7B5EA7"/><text x="118" y="295" font-size="13.5" fill="#FFFFFF" text-anchor="middle" font-weight="bold">2</text><rect x="175" y="264" width="150" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="250.0" y="290.0" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자료를</text><rect x="335" y="264" width="150" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="410.0" y="290.0" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">확인하고</text><line x1="493" y1="285" x2="647" y2="285" stroke="#8A93A0" stroke-width="2"/><polygon points="656,285 646,279 646,291" fill="#8A93A0"/><rect x="666" y="264" width="150" height="42" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="741.0" y="290.0" font-size="14.5" fill="#1D4726" text-anchor="middle" font-weight="bold">담당자에게</text><circle cx="118" cy="356" r="13" fill="#7B5EA7"/><text x="118" y="361" font-size="13.5" fill="#FFFFFF" text-anchor="middle" font-weight="bold">3</text><rect x="175" y="330" width="150" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="250.0" y="356.0" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자료를</text><rect x="335" y="330" width="150" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="410.0" y="356.0" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">확인하고</text><rect x="495" y="330" width="150" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="570.0" y="356.0" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">담당자에게</text><line x1="653" y1="351" x2="647" y2="351" stroke="#8A93A0" stroke-width="2"/><polygon points="656,351 646,345 646,357" fill="#8A93A0"/><rect x="666" y="330" width="150" height="42" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="741.0" y="356.0" font-size="14.5" fill="#1D4726" text-anchor="middle" font-weight="bold">전달했습니다</text><rect x="96" y="412" width="748" height="78" rx="12" ry="12" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="470.0" y="440" font-size="14.5" fill="#77400F" text-anchor="middle" font-weight="bold">정답은 사람이 따로 적어 주지 않습니다. 문장 안의 “바로 다음 단어” 가 그대로 정답이 됩니다.</text><text x="470.0" y="466" font-size="13.5" fill="#77400F" text-anchor="middle" font-weight="normal">그래서 문장만 있으면 학습 데이터가 자동으로 만들어집니다.</text><text x="470.0" y="522" font-size="14" fill="#3C4552" text-anchor="middle" font-weight="bold">입력이 한 단어 늘어날 때마다, 모델은 그 시점까지의 상태로 다음 단어를 예측합니다.</text><text x="470.0" y="544" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">①②③ 은 모두 같은 문장에서 자동으로 만들어진 연습 문제입니다.</text></svg>


### 그림에서 확인할 점
- 토큰 4개짜리 문장 하나에서 학습 문제 **3개**가 만들어집니다.
- 파란색이 모델에게 보여 줄 **입력**, 초록색이 맞혀야 할 **정답**입니다.
- 입력이 한 토큰 길어질 때마다 정답도 한 칸씩 뒤로 이동합니다.

In [4]:
# ============================================================
# [핵심 코드] 문장 → 다음 단어 학습 문제
#
#  입력   : 문장 목록 (corpus)
#  하는 일 : 문장마다 앞에서부터 토큰을 하나씩 늘려 가며
#            "지금까지 읽은 토큰 → 바로 다음 토큰" 문제를 만듭니다.
#  왜     : 다음 단어 예측은 정답을 사람이 따로 적어 줄 필요가 없습니다.
#  출력   : X (입력 토큰 배열), y (정답 토큰 번호)
#  다음   : 이 X, y 를 그대로 SimpleRNN / LSTM 학습에 사용합니다.
# ============================================================
def 다음단어_데이터_만들기(문장들):
    """문장 목록 → (입력 X, 정답 y)"""
    X, y = [], []

    for 문장 in 문장들:
        번호 = [int(t) for t in vectorize(np.array([문장], dtype=object)).numpy()[0] if t != 0]

        # 앞에서부터 한 단어씩 늘려 가며 문제를 만듭니다.
        for i in range(1, len(번호)):
            X.append(번호[:i])   # 지금까지 읽은 단어들 = 입력
            y.append(번호[i])    # 바로 다음 단어      = 정답

    # 모든 입력을 같은 길이로 맞춥니다. 빈칸은 앞쪽(pre)에 채웁니다.
    X = tf.keras.utils.pad_sequences(X, maxlen=MAX_LEN, padding="pre")
    return np.array(X), np.array(y)


X, y = 다음단어_데이터_만들기(corpus)

print("만들어진 학습 문제 수 :", len(X), "개")
print("입력 X 의 모양        :", X.shape, " ← (문제 수, 입력 칸 수)")
print("정답 y 의 모양        :", y.shape)
print()

만들어진 학습 문제 수 : 158 개
입력 X 의 모양        : (158, 13)  ← (문제 수, 입력 칸 수)
정답 y 의 모양        : (158,)


문장 하나가 실제로 어떤 문제들로 바뀌었는지 눈으로 확인합니다.

In [5]:
# ============================================================
# [결과 확인용 보조 코드 — RNN/LSTM 핵심 알고리즘이 아닙니다]
#
# 아래 코드는 만들어진 학습 문제를 눈으로 확인하기 위한 출력 코드입니다.
# Python 문법을 한 줄씩 분석할 필요는 없습니다.
# ============================================================
# 문장 하나가 어떻게 문제로 바뀌었는지 눈으로 확인합니다.
원문 = corpus[0]
번호 = [int(t) for t in vectorize(np.array([원문], dtype=object)).numpy()[0] if t != 0]

print("원문 :", 원문)
print()
print("이 문장에서 만들어진 학습 문제")
for i in range(1, len(번호)):
    입력 = " ".join(어휘[t] for t in 번호[:i])
    정답 = 어휘[번호[i]]
    print(f"  {입력}")
    print(f"      → {정답}")

원문 : 자료를 확인하고 담당자에게 전달했습니다

이 문장에서 만들어진 학습 문제
  자료를
      → 확인하고
  자료를 확인하고
      → 담당자에게
  자료를 확인하고 담당자에게
      → 전달했습니다


---
## 04-2. Padding — 왜 앞쪽에 빈칸을 넣나?

문장마다 토큰 수가 다르면 모델이 여러 문장을 한 번에 계산할 수 없습니다.
그래서 **모든 입력을 같은 길이로 맞춥니다.**


<svg viewBox="0 0 940 392" width="100%" role="img" aria-label="길이가 다른 두 문장의 앞쪽을 빈칸으로 채워 같은 길이로 맞추는 방법" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>앞쪽 Padding</title><desc>토큰 세 개짜리 문장은 앞의 두 칸을 빈칸으로 채우고 토큰 다섯 개짜리 문장은 빈칸 없이 채워, 두 문장 모두 다섯 칸이 되면서 실제 토큰이 항상 오른쪽 끝에 오도록 맞추는 모습을 보여 주는 그림</desc><rect x="0" y="0" width="940" height="392" fill="#FFFFFF"/><text x="470.0" y="34.0" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">길이를 맞추되, 실제 토큰은 항상 오른쪽 끝에 둡니다</text><text x="470.0" y="58.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">빈칸은 앞쪽에 넣습니다 (padding="pre")</text><text x="79.0" y="118.0" font-size="13.5" fill="#6B7280" text-anchor="start" font-weight="bold">토큰 3개짜리 문장</text><rect x="79.0" y="130.0" width="150.0" height="46.0" rx="10" ry="10" fill="#FFFFFF" stroke="#CBD0D8" stroke-width="1.5" stroke-dasharray="5 4"/><text x="154.0" y="159.0" font-size="13.5" fill="#6B7280" text-anchor="middle" font-weight="normal">빈칸 0</text><rect x="237.0" y="130.0" width="150.0" height="46.0" rx="10" ry="10" fill="#FFFFFF" stroke="#CBD0D8" stroke-width="1.5" stroke-dasharray="5 4"/><text x="312.0" y="159.0" font-size="13.5" fill="#6B7280" text-anchor="middle" font-weight="normal">빈칸 0</text><rect x="395.0" y="130.0" width="150.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="470.0" y="157.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자료를</text><rect x="553.0" y="130.0" width="150.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="628.0" y="157.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">확인하고</text><rect x="711.0" y="130.0" width="150.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="786.0" y="157.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">담당자에게</text><text x="79.0" y="228.0" font-size="13.5" fill="#6B7280" text-anchor="start" font-weight="bold">토큰 5개짜리 문장</text><rect x="79.0" y="240.0" width="150.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="154.0" y="267.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">관련</text><rect x="237.0" y="240.0" width="150.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="312.0" y="267.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자료를</text><rect x="395.0" y="240.0" width="150.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="470.0" y="267.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">검토하고</text><rect x="553.0" y="240.0" width="150.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="628.0" y="267.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">결과를</text><rect x="711.0" y="240.0" width="150.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="786.0" y="267.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">보고했습니다</text><line x1="866.0" y1="320.0" x2="866.0" y2="303.0" stroke="#4E9A57" stroke-width="2.0"/><polygon points="866.0,294.0 860.0,303.0 872.0,303.0" fill="#4E9A57"/><text x="866.0" y="340.0" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="bold">마지막 토큰</text><text x="866.0" y="358.0" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal">항상 맨 끝</text><text x="470.0" y="320.0" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="bold">이번 실습에서는 실제 토큰이 항상 오른쪽에 오도록 앞쪽을 0(PAD)으로 채웁니다.</text><text x="470.0" y="344.0" font-size="13.0" fill="#6B7280" text-anchor="middle" font-weight="normal">그러면 마지막 시점의 상태를 그대로 다음 단어 예측에 쓸 수 있습니다.</text></svg>


### 그림에서 확인할 점
- 위 그림은 칸이 5개인 예시입니다. 이번 실습에서 실제로 쓰는 칸 수는 `MAX_LEN` 이며, 03장 실행 결과에 나와 있습니다.
- 빈칸은 **앞쪽**에 넣습니다 (`padding="pre"`).
- 그래서 **실제 토큰은 항상 오른쪽 끝**에 옵니다.

### 왜 앞쪽인가요?

우리가 풀려는 문제가 **다음 단어 예측**이기 때문입니다.
마지막 실제 토큰이 항상 맨 끝에 오면, **마지막 시점의 상태**를 그대로 다음 단어 예측에 쓸 수 있습니다.

또한 `Embedding(..., mask_zero=True)` 로 지정했기 때문에 **빈칸(0번) 시점은 계산에서 건너뜁니다.**

---
# 05. SimpleRNN

**단어 토큰 → Embedding → ★ SimpleRNN (지금 여기) → Dense → 다음 단어 후보**

## 05-1. Hidden State 란 무엇인가?

> **Hidden State (은닉 상태, `h`)** — 이전까지 처리한 입력의 정보를 **숫자 벡터 형태로 요약한 상태**입니다.

### 매우 중요합니다

> ### Hidden State 안에 단어가 그대로 저장되는 것은 아닙니다.

`h` 는 사람의 기억과 같은 것이 아니라, **이전 계산 결과를 담은 숫자 벡터**입니다.
칸 하나하나를 특정 의미와 연결해서 해석하지 않습니다.

### 시점마다 만들어지는 상태

| 상태 | 무엇을 요약한 것인가 |
|---|---|
| **h₁** | `자료를` 까지 처리한 결과를 요약한 숫자 상태 |
| **h₂** | `자료를 확인하고` 까지 처리한 결과를 요약한 숫자 상태 |
| **h₃** | `자료를 확인하고 담당자에게` 까지 처리한 결과를 요약한 숫자 상태 |

“h₂ 안에 자료를 과 확인하고 라는 단어가 들어 있다”가 아니라,
**“그 두 토큰을 순서대로 처리한 결과가 숫자로 요약되어 있다”** 는 뜻입니다.

---
## 05-2. RNN 은 상태를 어떻게 전달하나?

각 시점에서 SimpleRNN 은 **두 가지를 함께 받습니다.**

| 받는 것 | 무엇인가 | 어디에서 오나 |
|---|---|---|
| **xₜ** | 이번 시점의 입력 벡터 | 현재 단어 토큰 → Embedding |
| **hₜ₋₁** | 직전 시점의 Hidden State | 바로 앞 시점의 RNN |

이 둘을 함께 사용해 **새 상태 hₜ** 를 만들고, 그 hₜ 를 다시 다음 시점으로 넘깁니다.


<svg viewBox="0 0 940 600" width="100%" role="img" aria-label="SimpleRNN을 시간축으로 펼쳐 Hidden State가 다음 시점으로 전달되는 구조" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>SimpleRNN 시간축</title><desc>자료를 확인하고 담당자에게 세 개의 단어 토큰이 각각 Embedding을 거쳐 입력 벡터가 되고, 같은 SimpleRNN 층이 세 시점에서 반복 사용되면서 Hidden State가 왼쪽에서 오른쪽으로 전달되며, 마지막 Hidden State가 Dense와 Softmax를 거쳐 다음 단어 후보가 되는 흐름을 가로로 펼쳐 그린 그림</desc><rect x="0" y="0" width="940" height="600" fill="#FFFFFF"/><text x="470.0" y="34.0" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">같은 SimpleRNN 이 시점마다 반복되며 상태를 전달합니다</text><text x="470.0" y="58.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">시간은 왼쪽에서 오른쪽으로 흐릅니다</text><text x="190.0" y="82.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="bold">시점 t1</text><rect x="100.0" y="92.0" width="180.0" height="42.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="190.0" y="118.2" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자료를</text><line x1="190.0" y1="134.0" x2="190.0" y2="147.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="190.0,156.0 196.0,147.0 184.0,147.0" fill="#8A93A0"/><rect x="100.0" y="156.0" width="180.0" height="42.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="190.0" y="181.9" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">Embedding</text><line x1="190.0" y1="198.0" x2="190.0" y2="223.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="190.0,232.0 196.0,223.0 184.0,223.0" fill="#8A93A0"/><text x="212.0" y="220.0" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="bold">x<tspan font-size="9.7" dy="3">1</tspan></text><rect x="100.0" y="232.0" width="180.0" height="78.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="190.0" y="268.0" font-size="16" fill="#3B295D" text-anchor="middle" font-weight="bold">SimpleRNN</text><text x="190.0" y="288.0" font-size="13.6" fill="#3B295D" text-anchor="middle" font-weight="normal" opacity="0.85">같은 층 · 같은 가중치</text><text x="450.0" y="82.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="bold">시점 t2</text><rect x="360.0" y="92.0" width="180.0" height="42.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="450.0" y="118.2" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">확인하고</text><line x1="450.0" y1="134.0" x2="450.0" y2="147.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="450.0,156.0 456.0,147.0 444.0,147.0" fill="#8A93A0"/><rect x="360.0" y="156.0" width="180.0" height="42.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="450.0" y="181.9" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">Embedding</text><line x1="450.0" y1="198.0" x2="450.0" y2="223.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="450.0,232.0 456.0,223.0 444.0,223.0" fill="#8A93A0"/><text x="472.0" y="220.0" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="bold">x<tspan font-size="9.7" dy="3">2</tspan></text><rect x="360.0" y="232.0" width="180.0" height="78.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="450.0" y="268.0" font-size="16" fill="#3B295D" text-anchor="middle" font-weight="bold">SimpleRNN</text><text x="450.0" y="288.0" font-size="13.6" fill="#3B295D" text-anchor="middle" font-weight="normal" opacity="0.85">같은 층 · 같은 가중치</text><text x="710.0" y="82.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="bold">시점 t3</text><rect x="620.0" y="92.0" width="180.0" height="42.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="710.0" y="118.2" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">담당자에게</text><line x1="710.0" y1="134.0" x2="710.0" y2="147.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="710.0,156.0 716.0,147.0 704.0,147.0" fill="#8A93A0"/><rect x="620.0" y="156.0" width="180.0" height="42.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="710.0" y="181.9" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">Embedding</text><line x1="710.0" y1="198.0" x2="710.0" y2="223.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="710.0,232.0 716.0,223.0 704.0,223.0" fill="#8A93A0"/><text x="732.0" y="220.0" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="bold">x<tspan font-size="9.7" dy="3">3</tspan></text><rect x="620.0" y="232.0" width="180.0" height="78.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="710.0" y="268.0" font-size="16" fill="#3B295D" text-anchor="middle" font-weight="bold">SimpleRNN</text><text x="710.0" y="288.0" font-size="13.6" fill="#3B295D" text-anchor="middle" font-weight="normal" opacity="0.85">같은 층 · 같은 가중치</text><line x1="24.0" y1="271.0" x2="92.0" y2="271.0" stroke="#CBD0D8" stroke-width="2.0" stroke-dasharray="4 3"/><line x1="92.0" y1="271.0" x2="93.0" y2="271.0" stroke="#CBD0D8" stroke-width="2.0"/><polygon points="100.0,271.0 93.0,266.0 93.0,276.0" fill="#CBD0D8"/><text x="56.0" y="261.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">h<tspan font-size="9.4" dy="3">0</tspan></text><line x1="280.0" y1="271.0" x2="351.0" y2="271.0" stroke="#7B5EA7" stroke-width="2.6"/><polygon points="360.0,271.0 351.0,265.0 351.0,277.0" fill="#7B5EA7"/><text x="320.0" y="260.0" font-size="14" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.1" dy="3">1</tspan></text><line x1="540.0" y1="271.0" x2="611.0" y2="271.0" stroke="#7B5EA7" stroke-width="2.6"/><polygon points="620.0,271.0 611.0,265.0 611.0,277.0" fill="#7B5EA7"/><text x="580.0" y="260.0" font-size="14" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.1" dy="3">2</tspan></text><line x1="800.0" y1="271.0" x2="700.0" y2="271.0" stroke="#7B5EA7" stroke-width="0.0"/><polygon points="700.0,271.0 700.0,271.0 700.0,271.0" fill="#7B5EA7"/><line x1="710.0" y1="310.0" x2="710.0" y2="335.0" stroke="#7B5EA7" stroke-width="2.6"/><polygon points="710.0,344.0 716.0,335.0 704.0,335.0" fill="#7B5EA7"/><text x="738.0" y="334.0" font-size="14" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.1" dy="3">3</tspan></text><rect x="570.0" y="344.0" width="280.0" height="52.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="710.0" y="367.0" font-size="15" fill="#0E4A46" text-anchor="middle" font-weight="bold">Dense + Softmax</text><text x="710.0" y="387.0" font-size="12.75" fill="#0E4A46" text-anchor="middle" font-weight="normal" opacity="0.85">어휘 전체 점수 계산</text><line x1="710.0" y1="396.0" x2="710.0" y2="415.0" stroke="#7B5EA7" stroke-width="2.6"/><polygon points="710.0,424.0 716.0,415.0 704.0,415.0" fill="#7B5EA7"/><rect x="570.0" y="424.0" width="280.0" height="52.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="710.0" y="447.0" font-size="15" fill="#1D4726" text-anchor="middle" font-weight="bold">다음 단어 후보</text><text x="710.0" y="467.0" font-size="12.75" fill="#1D4726" text-anchor="middle" font-weight="normal" opacity="0.85">전달했습니다 · 공유했습니다 …</text><rect x="40.0" y="344.0" width="480.0" height="132.0" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="64.0" y="372.0" font-size="14.5" fill="#555C68" text-anchor="start" font-weight="bold">그림 보는 법</text><text x="72.0" y="400.0" font-size="15" fill="#0E4A46" text-anchor="start" font-weight="bold">x</text><text x="98.0" y="400.0" font-size="13" fill="#3C4552" text-anchor="start" font-weight="normal">Embedding 을 거친 이번 시점의 입력 벡터</text><text x="72.0" y="426.0" font-size="15" fill="#7B5EA7" text-anchor="start" font-weight="bold">h</text><text x="98.0" y="426.0" font-size="13" fill="#3C4552" text-anchor="start" font-weight="normal">지금까지 처리한 입력의 영향을 요약한 상태</text><text x="72.0" y="452.0" font-size="15" fill="#3B295D" text-anchor="start" font-weight="bold">=</text><text x="98.0" y="452.0" font-size="13" fill="#3C4552" text-anchor="start" font-weight="normal">네모 세 개는 같은 SimpleRNN 이 반복된 것</text><text x="470.0" y="514.0" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="bold">그림에서는 시간축으로 펼쳐 세 번 그렸지만, 실제로는 같은 SimpleRNN 층의 같은 가중치를 반복해서 사용합니다.</text><text x="470.0" y="538.0" font-size="13.0" fill="#6B7280" text-anchor="middle" font-weight="normal">h 안에 단어가 그대로 저장되는 것이 아니라, 지금까지 처리한 입력의 영향이 숫자 벡터로 요약되어 있습니다.</text></svg>


### 그림에서 확인할 점

- 시간은 **왼쪽에서 오른쪽**으로 흐릅니다.
- 단어 토큰은 곧바로 RNN 에 들어가지 않습니다. **먼저 Embedding 을 거쳐 입력 벡터 `x` 가 됩니다.**
- 보라색 화살표(`h₁`, `h₂`)가 **이번 실습에서 가장 중요한 부분**입니다.

> **그림에서는 시간축으로 펼쳐 세 번 그렸지만,
> 실제로는 같은 SimpleRNN 층의 같은 가중치를 반복해서 사용합니다.**

그래서 문장이 길어져도 **모델의 크기(파라미터 수)는 늘어나지 않습니다.**

---
## 05-3. 마지막 Hidden State 는 어떻게 다음 단어가 되나?

마지막 토큰까지 읽고 나면 상태 `h` 하나가 남습니다.
이 `h` 는 **문장 앞부분 전체를 순서대로 처리한 결과**입니다.

| 순서 | 단계 | 하는 일 |
|---|---|---|
| 1 | **마지막 Hidden State** | 문장 앞부분 전체를 요약한 숫자 벡터 |
| 2 | **Dense** | 사전에 있는 모든 단어 토큰에 대해 점수를 계산 |
| 3 | **Softmax** | 점수를 서로 비교할 수 있는 값으로 정리 |
| 4 | **다음 단어 후보** | 점수가 높은 순서대로 후보 제시 |

즉 다음 단어는 **마지막 상태 하나**에서 계산됩니다.
그래서 앞에서 어떤 토큰을 어떤 순서로 읽었는지가 결과를 바꿉니다.

---
## 05-4. 실제 코드에서는 어떻게 보이나?

| 층 | 코드 | 하는 일 |
|---|---|---|
| 1 | `Embedding(VOCAB_SIZE, 32, mask_zero=True)` | Token ID → 숫자 벡터 |
| 2 | `SimpleRNN(64)` | ★ 순서대로 읽으며 상태를 전달 |
| 3 | `Dense(VOCAB_SIZE, activation="softmax")` | 다음 단어 후보 점수 |

아래 셀의 주석과 함께 위에서 아래로 읽으면 구조가 그대로 보입니다.

> **선택 학습** — 층 목록과 파라미터 수를 확인하고 싶다면 `rnn_model.summary()` 를 따로 실행해 볼 수 있습니다.
> 이번 실습의 핵심은 파라미터 수가 아니라 **상태 전달**이므로 기본 출력에서는 뺐습니다.

In [6]:
rnn_model = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(MAX_LEN,)),

    # 단어 번호를 학습 가능한 숫자 벡터로 바꿉니다.
    # 아직 RNN 자체는 아닙니다.
    tf.keras.layers.Embedding(
        VOCAB_SIZE,
        32,
        mask_zero=True,      # 앞쪽 빈칸(0번)은 계산에서 건너뜁니다.
    ),

    # ==================================================
    # ★ RNN 핵심
    #
    # 문장을 앞에서부터 한 단어씩 읽습니다.
    #
    # 현재 단어의 정보와
    # 이전 시점의 Hidden State 를 함께 사용하여
    # 새로운 Hidden State 를 만듭니다.
    #
    # 이 Hidden State 가 다시 다음 단어 처리에 전달됩니다.
    # ==================================================
    tf.keras.layers.SimpleRNN(64),

    # 마지막 Hidden State 를 이용하여
    # 다음 단어별 후보 점수를 계산합니다.
    tf.keras.layers.Dense(
        VOCAB_SIZE,
        activation="softmax",
    ),
])

---
# 06. SimpleRNN 학습

## 06-1. 학습시키기

문장 28개에서 만든 학습 문제를 여러 번 반복해서 보여 줍니다.
**Loss(손실)가 줄어들었다면 학습이 진행된 것입니다.**

In [7]:
# ============================================================
# [핵심 코드] SimpleRNN 학습
#
#  입력  : X (다음 단어 문제), y (정답 토큰 번호)
#  하는 일: 예측이 정답에 가까워지도록 Embedding · SimpleRNN · Dense 의
#           가중치를 EPOCHS 번 반복해서 조금씩 고칩니다.
#  출력  : 학습이 끝난 rnn_model 과 Loss 기록
#  다음  : 06-2 에서 이 모델에 문장 앞부분을 넣어 다음 단어를 예측합니다.
# ============================================================
EPOCHS = 300   # 문장이 매우 적으므로 여러 번 반복해서 봅니다.

rnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
)

rnn_history = rnn_model.fit(
    X, y,
    epochs=EPOCHS,
    validation_split=0.2,
    verbose=0,
)

print(
    f"RNN Loss: "
    f"{rnn_history.history['loss'][0]:.3f}"
    f" → {rnn_history.history['loss'][-1]:.3f}"
)
print("→ Loss(손실)가 줄어들었다면 학습이 진행된 것입니다.")

RNN Loss: 4.308 → 0.057
→ Loss(손실)가 줄어들었다면 학습이 진행된 것입니다.


---
## 06-2. 다음 단어를 예측해 봅니다

학습이 끝난 SimpleRNN 에 문장의 **앞부분**만 넣고, 다음에 올 단어 후보 **3개**를 봅니다.

### 후보 점수를 읽는 방법

아래 숫자는 **모델이 계산한 후보 점수**입니다. 다음과 같이 읽지 **않습니다.**

- ✕ “99% 확률로 정답입니다”
- ✕ “정답은 전달했습니다 입니다”

자연어에서는 같은 앞부분 뒤에 **여러 표현이 모두 자연스럽게** 올 수 있습니다.
AI 는 판단자가 아니라 **사람이 확인할 후보를 제안하는 보조자**입니다.

### 무엇을 확인하면 되나요?
- **입력이 달라지면 1순위 후보도 달라지는지**
- 이 모델이 “무슨 문장이든 항상 같은 단어”만 답하지는 않는지

In [8]:
# ============================================================
# [결과 확인용 보조 코드 — RNN/LSTM 핵심 알고리즘이 아닙니다]
#
# 아래 코드는 예측 결과를 보기 쉽게 출력하기 위한 준비 코드입니다.
# Python 문법을 한 줄씩 분석할 필요는 없습니다.
# ============================================================
def 문장을_숫자로(문장):
    """문장 → 모델 입력 (앞쪽을 빈칸으로 채운 길이 MAX_LEN 배열)"""
    번호 = [int(t) for t in vectorize(np.array([문장], dtype=object)).numpy()[0] if t != 0]
    return tf.keras.utils.pad_sequences([번호], maxlen=MAX_LEN, padding="pre")


def 다음단어_후보(모델, 문장, k=3):
    """문장 뒤에 올 단어 후보 k개를 (단어, 점수) 목록으로 돌려줍니다."""
    점수 = 모델.predict(문장을_숫자로(문장), verbose=0)[0].copy()

    점수[:2] = -1     # 0번(빈칸)과 1번([UNK])은 후보에서 제외합니다.

    상위 = np.argsort(점수)[-k:][::-1]
    return [(어휘[i], float(점수[i])) for i in 상위]

In [9]:
# ---- 교안 대표 예제를 포함한 서로 다른 문맥 ----
시연문장 = [
    "자료를 확인하고 담당자에게",          # ← 교안 대표 예제
    "회의 자료를 검토한 후 결과를",
    "신청서를 접수한 후 민원인에게",
]

for 문장 in 시연문장:
    print("=" * 60)
    print(f'입력 : "{문장} ___"')
    print("다음 단어 후보")
    for 순위, (단어, 점수) in enumerate(다음단어_후보(rnn_model, 문장), 1):
        print(f"  {순위}. {단어}  {점수:.3f}")
    print()

입력 : "자료를 확인하고 담당자에게 ___"
다음 단어 후보
  1. 전달했습니다  0.996
  2. 내용을  0.001
  3. 안내했습니다  0.000

입력 : "회의 자료를 검토한 후 결과를 ___"
다음 단어 후보
  1. 공유했습니다  0.994
  2. 서류를  0.002
  3. 내용을  0.001

입력 : "신청서를 접수한 후 민원인에게 ___"
다음 단어 후보
  1. 안내했습니다  0.996
  2. 뒤  0.001
  3. 전달했습니다  0.001



---
# 07. 순서 변경

## 07-1. 정상 순서 vs 변경 순서

**똑같은 단어 토큰**으로 이루어졌지만 **순서만 다른** 입력을 나란히 넣어 봅니다.


<svg viewBox="0 0 940 540" width="100%" role="img" aria-label="같은 단어를 다른 순서로 넣었을 때 Hidden State가 만들어지는 경로 비교" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>정상 순서와 변경 순서의 상태 경로</title><desc>위쪽은 자료를 확인하고 담당자에게 순서로, 아래쪽은 담당자에게 자료를 확인하고 순서로 같은 단어를 넣었을 때 Hidden State가 서로 다른 경로로 만들어지고 마지막 상태도 달라지는 과정을 가로로 비교한 그림</desc><rect x="0" y="0" width="940" height="540" fill="#FFFFFF"/><text x="470.0" y="34.0" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">같은 단어라도 순서가 다르면 상태 경로가 달라집니다</text><text x="470.0" y="58.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">들어간 단어는 완전히 같습니다</text><rect x="40.0" y="86.0" width="860.0" height="122.0" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="64.0" y="108.0" font-size="15" fill="#1B3A5E" text-anchor="start" font-weight="bold">정상 순서</text><rect x="100.0" y="124.0" width="148.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="174.0" y="151.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자료를</text><line x1="248.0" y1="147.0" x2="267.0" y2="147.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="276.0,147.0 267.0,141.0 267.0,153.0" fill="#8A93A0"/><circle cx="303.0" cy="147.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="303.0" y="153.0" font-size="15" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.8" dy="3">1</tspan></text><line x1="328.0" y1="147.0" x2="347.0" y2="147.0" stroke="#7B5EA7" stroke-width="2.2"/><polygon points="356.0,147.0 347.0,141.0 347.0,153.0" fill="#7B5EA7"/><rect x="356.0" y="124.0" width="148.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="430.0" y="151.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">확인하고</text><line x1="504.0" y1="147.0" x2="523.0" y2="147.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="532.0,147.0 523.0,141.0 523.0,153.0" fill="#8A93A0"/><circle cx="559.0" cy="147.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="559.0" y="153.0" font-size="15" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.8" dy="3">2</tspan></text><line x1="584.0" y1="147.0" x2="603.0" y2="147.0" stroke="#7B5EA7" stroke-width="2.2"/><polygon points="612.0,147.0 603.0,141.0 603.0,153.0" fill="#7B5EA7"/><rect x="612.0" y="124.0" width="148.0" height="46.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="686.0" y="151.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">담당자에게</text><line x1="760.0" y1="147.0" x2="779.0" y2="147.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="788.0,147.0 779.0,141.0 779.0,153.0" fill="#8A93A0"/><circle cx="815.0" cy="147.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="815.0" y="153.0" font-size="15" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.8" dy="3">3</tspan></text><text x="812.0" y="186.0" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="normal">마지막 상태</text><rect x="40.0" y="236.0" width="860.0" height="122.0" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="64.0" y="258.0" font-size="15" fill="#77400F" text-anchor="start" font-weight="bold">순서 변경</text><rect x="100.0" y="274.0" width="148.0" height="46.0" rx="10" ry="10" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="174.0" y="301.9" font-size="13.5" fill="#77400F" text-anchor="middle" font-weight="normal">담당자에게</text><line x1="248.0" y1="297.0" x2="267.0" y2="297.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="276.0,297.0 267.0,291.0 267.0,303.0" fill="#8A93A0"/><circle cx="303.0" cy="297.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="303.0" y="303.0" font-size="15" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.8" dy="3">1'</tspan></text><line x1="328.0" y1="297.0" x2="347.0" y2="297.0" stroke="#7B5EA7" stroke-width="2.2"/><polygon points="356.0,297.0 347.0,291.0 347.0,303.0" fill="#7B5EA7"/><rect x="356.0" y="274.0" width="148.0" height="46.0" rx="10" ry="10" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="430.0" y="301.9" font-size="13.5" fill="#77400F" text-anchor="middle" font-weight="normal">자료를</text><line x1="504.0" y1="297.0" x2="523.0" y2="297.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="532.0,297.0 523.0,291.0 523.0,303.0" fill="#8A93A0"/><circle cx="559.0" cy="297.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="559.0" y="303.0" font-size="15" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.8" dy="3">2'</tspan></text><line x1="584.0" y1="297.0" x2="603.0" y2="297.0" stroke="#7B5EA7" stroke-width="2.2"/><polygon points="612.0,297.0 603.0,291.0 603.0,303.0" fill="#7B5EA7"/><rect x="612.0" y="274.0" width="148.0" height="46.0" rx="10" ry="10" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="686.0" y="301.9" font-size="13.5" fill="#77400F" text-anchor="middle" font-weight="normal">확인하고</text><line x1="760.0" y1="297.0" x2="779.0" y2="297.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="788.0,297.0 779.0,291.0 779.0,303.0" fill="#8A93A0"/><circle cx="815.0" cy="297.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="815.0" y="303.0" font-size="15" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.8" dy="3">3'</tspan></text><text x="812.0" y="336.0" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="normal">마지막 상태</text><rect x="45.0" y="412.0" width="196.0" height="54.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="143.0" y="443.5" font-size="12.5" fill="#3B295D" text-anchor="middle" font-weight="normal">같은 단어 토큰, 다른 순서</text><line x1="241.0" y1="439.0" x2="255.0" y2="439.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="263.0,439.0 255.0,434.0 255.0,444.0" fill="#8A93A0"/><rect x="263.0" y="412.0" width="196.0" height="54.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="361.0" y="443.5" font-size="12.5" fill="#3B295D" text-anchor="middle" font-weight="normal">h 가 만들어지는 경로가 다름</text><line x1="459.0" y1="439.0" x2="473.0" y2="439.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="481.0,439.0 473.0,434.0 473.0,444.0" fill="#8A93A0"/><rect x="481.0" y="412.0" width="196.0" height="54.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="579.0" y="443.5" font-size="12.5" fill="#3B295D" text-anchor="middle" font-weight="normal">마지막 상태가 다름</text><line x1="677.0" y1="439.0" x2="691.0" y2="439.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="699.0,439.0 691.0,434.0 691.0,444.0" fill="#8A93A0"/><rect x="699.0" y="412.0" width="196.0" height="54.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="797.0" y="443.5" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal">다음 단어 후보도 달라질 수 있음</text><text x="470.0" y="496.0" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="bold">앞 실습처럼 각 토큰의 표현을 위치와 상관없이 평균으로 합치면,</text><text x="470.0" y="520.0" font-size="13.0" fill="#6B7280" text-anchor="middle" font-weight="normal">같은 표현이 같은 횟수만큼 들어 있는 경우 순서 차이가 평균값에 직접 반영되지 않습니다.</text></svg>


### 그림에서 확인할 점
- 두 줄에 들어간 단어 토큰은 **완전히 같습니다.**
- 그런데 `h` 가 만들어지는 **경로**가 다릅니다.
- 경로가 다르면 **마지막 상태**도 다르고, 다음 단어 후보도 달라질 수 있습니다.

In [10]:
순서비교 = [
    ("정상 순서", "자료를 확인하고 담당자에게"),
    ("순서 변경", "담당자에게 자료를 확인하고"),
]

for 이름, 문장 in 순서비교:
    print("=" * 60)
    print(f'[{이름}]  "{문장} ___"')
    for 순위, (단어, 점수) in enumerate(다음단어_후보(rnn_model, 문장), 1):
        print(f"  {순위}. {단어}  {점수:.3f}")
    print()

print("→ 두 입력에 들어 있는 단어는 완전히 같습니다.")
print("→ 그런데 읽은 순서가 다르므로 전달되는 상태가 달라지고, 후보도 달라집니다.")

[정상 순서]  "자료를 확인하고 담당자에게 ___"
  1. 전달했습니다  0.996
  2. 내용을  0.001
  3. 안내했습니다  0.000

[순서 변경]  "담당자에게 자료를 확인하고 ___"
  1. 검토한  0.354
  2. 정리한  0.250
  3. 담당자에게  0.185

→ 두 입력에 들어 있는 단어는 완전히 같습니다.
→ 그런데 읽은 순서가 다르므로 전달되는 상태가 달라지고, 후보도 달라집니다.


---
## 07-2. 왜 마지막 h 가 달라지는가

RNN 은 매 시점에서 **직전 상태**를 함께 사용합니다.
그래서 어떤 토큰을 **몇 번째로** 읽었는지가 그대로 상태에 영향을 줍니다.

| | 정상 순서 | 순서 변경 |
|---|---|---|
| 1번째 | `자료를` → h₁ | `담당자에게` → h₁′ |
| 2번째 | `확인하고` + h₁ → h₂ | `자료를` + h₁′ → h₂′ |
| 3번째 | `담당자에게` + h₂ → h₃ | `확인하고` + h₂′ → h₃′ |
| 마지막 상태 | **h₃** | **h₃′** |

h₁ 과 h₁′ 이 이미 다르므로, 그 뒤의 모든 상태가 달라집니다.

> 앞 실습처럼 각 토큰의 표현을 **위치와 상관없이 평균으로 합치면**,
> 같은 표현이 같은 횟수만큼 들어 있는 경우 **순서 차이가 평균값에 직접 반영되지 않습니다.**

---
# 08. 왜 LSTM 이 필요한가

**단어 토큰 → Embedding → ★ LSTM (지금 여기) → Dense → 다음 단어 후보**

## 08-1. SimpleRNN 은 왜 긴 문맥에서 어려울 수 있나?

SimpleRNN 은 상태 `h` **하나**만 계속 갱신하며 앞으로 나아갑니다.


<svg viewBox="0 0 940 440" width="100%" role="img" aria-label="SimpleRNN에서 초기 정보의 영향이 시점을 지나며 약해질 수 있음을 나타낸 개념 그림" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>멀리 있는 정보의 영향</title><desc>문장 맨 앞의 중요한 정보가 h1부터 h8까지 여덟 시점을 지나 마지막 예측에 도달하는 동안 연결선이 점점 옅어지도록 그려, 멀리 떨어진 정보의 영향을 안정적으로 학습하기가 어려워질 수 있다는 개념을 나타낸 그림</desc><rect x="0" y="0" width="940" height="440" fill="#FFFFFF"/><text x="470.0" y="34.0" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">멀리 떨어진 정보일수록 영향을 학습하기 어려워질 수 있습니다</text><text x="470.0" y="58.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">개념을 나타낸 그림입니다 · 실제 값이 일정하게 줄어든다는 뜻은 아닙니다</text><rect x="40.0" y="96.0" width="216.0" height="52.0" rx="10" ry="10" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="148.0" y="126.9" font-size="13.5" fill="#77400F" text-anchor="middle" font-weight="normal">문장 앞의 중요한 정보</text><line x1="148.0" y1="148.0" x2="148.0" y2="187.0" stroke="#DE8A3E" stroke-width="2.4"/><polygon points="148.0,196.0 154.0,187.0 142.0,187.0" fill="#DE8A3E"/><circle cx="148.0" cy="222.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="148.0" y="228.0" font-size="14.5" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.4" dy="3">1</tspan></text><line x1="173.0" y1="222.0" x2="210.0" y2="222.0" stroke="#7B5EA7" stroke-width="2.4" opacity="0.82"/><polygon points="219.0,222.0 210.0,216.0 210.0,228.0" fill="#7B5EA7" opacity="0.82"/><circle cx="244.0" cy="222.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="244.0" y="228.0" font-size="14.5" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.4" dy="3">2</tspan></text><line x1="269.0" y1="222.0" x2="306.0" y2="222.0" stroke="#7B5EA7" stroke-width="2.4" opacity="0.72"/><polygon points="315.0,222.0 306.0,216.0 306.0,228.0" fill="#7B5EA7" opacity="0.72"/><circle cx="340.0" cy="222.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="340.0" y="228.0" font-size="14.5" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.4" dy="3">3</tspan></text><line x1="365.0" y1="222.0" x2="402.0" y2="222.0" stroke="#7B5EA7" stroke-width="2.4" opacity="0.62"/><polygon points="411.0,222.0 402.0,216.0 402.0,228.0" fill="#7B5EA7" opacity="0.62"/><circle cx="436.0" cy="222.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="436.0" y="228.0" font-size="14.5" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.4" dy="3">4</tspan></text><line x1="461.0" y1="222.0" x2="498.0" y2="222.0" stroke="#7B5EA7" stroke-width="2.4" opacity="0.52"/><polygon points="507.0,222.0 498.0,216.0 498.0,228.0" fill="#7B5EA7" opacity="0.52"/><circle cx="532.0" cy="222.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="532.0" y="228.0" font-size="14.5" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.4" dy="3">5</tspan></text><line x1="557.0" y1="222.0" x2="594.0" y2="222.0" stroke="#7B5EA7" stroke-width="2.4" opacity="0.42"/><polygon points="603.0,222.0 594.0,216.0 594.0,228.0" fill="#7B5EA7" opacity="0.42"/><circle cx="628.0" cy="222.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="628.0" y="228.0" font-size="14.5" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.4" dy="3">6</tspan></text><line x1="653.0" y1="222.0" x2="690.0" y2="222.0" stroke="#7B5EA7" stroke-width="2.4" opacity="0.32"/><polygon points="699.0,222.0 690.0,216.0 690.0,228.0" fill="#7B5EA7" opacity="0.32"/><circle cx="724.0" cy="222.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="724.0" y="228.0" font-size="14.5" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.4" dy="3">7</tspan></text><line x1="749.0" y1="222.0" x2="786.0" y2="222.0" stroke="#7B5EA7" stroke-width="2.4" opacity="0.22"/><polygon points="795.0,222.0 786.0,216.0 786.0,228.0" fill="#7B5EA7" opacity="0.22"/><circle cx="820.0" cy="222.0" r="25.0" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="2.2"/><text x="820.0" y="228.0" font-size="14.5" fill="#7B5EA7" text-anchor="middle" font-weight="bold">h<tspan font-size="10.4" dy="3">8</tspan></text><text x="470.0" y="282.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">시점을 지날수록 앞쪽 정보의 영향이 옅어질 수 있습니다</text><line x1="844.0" y1="247.0" x2="844.0" y2="283.0" stroke="#7B5EA7" stroke-width="2.4" opacity="0.30"/><polygon points="844.0,292.0 850.0,283.0 838.0,283.0" fill="#7B5EA7" opacity="0.30"/><rect x="700.0" y="292.0" width="200.0" height="50.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="800.0" y="322.0" font-size="14" fill="#1D4726" text-anchor="middle" font-weight="normal">마지막 예측</text><text x="470.0" y="378.0" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="bold">문장이 길어질수록 멀리 떨어진 정보의 영향을 안정적으로 학습하기 어려워질 수 있습니다.</text><text x="470.0" y="402.0" font-size="13.0" fill="#6B7280" text-anchor="middle" font-weight="normal">반드시 잊는다는 뜻이 아니라, 학습이 어려워질 수 있다는 의미입니다.</text></svg>


### 정확하게 이해하기

> **문장이 길어질수록 멀리 떨어진 정보의 영향을 안정적으로 학습하기 어려워질 수 있습니다.**

- ✕ “긴 문장을 **처리하지 못한다**” — 아닙니다. 처리는 됩니다.
- ✕ “앞의 정보를 **반드시 잊는다**” — 아닙니다. 항상 잊는 것은 아닙니다.
- ○ **멀리 떨어진 정보 사이의 관계를 학습하기 어려워질 수 있다**는 의미입니다.

02-2 장에서 만든 긴 문장 세 개가 바로 이 상황입니다.
마지막 단어를 맞히려면 **여덟 단어 앞의 도입 부분**을 끝까지 유지해야 합니다.

---
## 08-2. Cell State

### LSTM 이 추가한 것

| | SimpleRNN | LSTM |
|---|---|---|
| 다음 시점으로 넘기는 것 | `h` 하나 | **`h` 와 `c` 두 개** |
| 정보 조절 | 없음 | **Gate 세 개로 조절** |
| 설계 의도 | 순서대로 읽기 | **필요한 정보를 더 오래 전달하기** |

### Cell State 란?

> **Cell State (`c`)** — 필요한 정보를 더 오래 전달하기 위해 LSTM 내부에서 사용하는 **별도의 상태 통로**입니다.

이해를 돕기 위해 ‘장기 정보 통로’라고 비유할 수 있지만, **실제로는 숫자 벡터**입니다.
사람의 장기 기억과 같은 것이 아닙니다.


<svg viewBox="0 0 940 640" width="100%" role="img" aria-label="LSTM이 Hidden State와 Cell State 두 개의 상태를 다음 시점으로 전달하는 구조" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>LSTM 의 두 통로</title><desc>현재 단어 토큰이 Embedding을 거쳐 입력 벡터가 되어 LSTM에 들어가고, LSTM은 이전 Hidden State와 이전 Cell State를 함께 받아 새 Hidden State와 새 Cell State 두 개를 다음 시점으로 내보내는 구조를 가로로 그린 그림. Cell State는 굵은 주황 이중선으로, Hidden State는 보라색 실선으로 구분해 표시</desc><rect x="0" y="0" width="940" height="640" fill="#FFFFFF"/><text x="470.0" y="34.0" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">LSTM 은 상태를 두 개 전달합니다</text><text x="470.0" y="58.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">Hidden State h 와 Cell State c 를 함께 다음 시점으로 넘깁니다</text><rect x="340.0" y="130.0" width="260.0" height="210.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="470.0" y="214.0" font-size="20" fill="#3B295D" text-anchor="middle" font-weight="bold">LSTM</text><text x="470.0" y="238.0" font-size="12.5" fill="#3B295D" text-anchor="middle" font-weight="normal" opacity="0.85">Gate 가 정보량을 조절</text><rect x="360.0" y="252.0" width="66.0" height="30.0" rx="8" ry="8" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="1.2"/><text x="393.0" y="272.0" font-size="11.5" fill="#3B295D" text-anchor="middle" font-weight="normal">Forget</text><rect x="434.0" y="252.0" width="66.0" height="30.0" rx="8" ry="8" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="1.2"/><text x="467.0" y="272.0" font-size="11.5" fill="#3B295D" text-anchor="middle" font-weight="normal">Input</text><rect x="508.0" y="252.0" width="66.0" height="30.0" rx="8" ry="8" fill="#FFFFFF" stroke="#7B5EA7" stroke-width="1.2"/><text x="541.0" y="272.0" font-size="11.5" fill="#3B295D" text-anchor="middle" font-weight="normal">Output</text><line x1="40.0" y1="165.0" x2="340.0" y2="165.0" stroke="#DE8A3E" stroke-width="2.2"/><line x1="600.0" y1="165.0" x2="890.0" y2="165.0" stroke="#DE8A3E" stroke-width="2.2"/><line x1="40.0" y1="171.0" x2="340.0" y2="171.0" stroke="#DE8A3E" stroke-width="2.2"/><line x1="600.0" y1="171.0" x2="890.0" y2="171.0" stroke="#DE8A3E" stroke-width="2.2"/><line x1="330.0" y1="168.0" x2="330.0" y2="168.0" stroke="#DE8A3E" stroke-width="2.2"/><polygon points="340.0,168.0 330.0,161.0 330.0,175.0" fill="#DE8A3E"/><line x1="880.0" y1="168.0" x2="890.0" y2="168.0" stroke="#DE8A3E" stroke-width="2.2"/><polygon points="900.0,168.0 890.0,161.0 890.0,175.0" fill="#DE8A3E"/><text x="72.0" y="152.0" font-size="14" fill="#DE8A3E" text-anchor="start" font-weight="bold">c<tspan font-size="10.1" dy="3">t-1</tspan></text><text x="840.0" y="152.0" font-size="14" fill="#DE8A3E" text-anchor="start" font-weight="bold">c<tspan font-size="10.1" dy="3">t</tspan></text><text x="190.0" y="200.0" font-size="12.5" fill="#DE8A3E" text-anchor="middle" font-weight="normal">이전 Cell State</text><text x="745.0" y="200.0" font-size="12.5" fill="#DE8A3E" text-anchor="middle" font-weight="normal">새 Cell State</text><line x1="40.0" y1="302.0" x2="331.0" y2="302.0" stroke="#7B5EA7" stroke-width="2.6"/><polygon points="340.0,302.0 331.0,296.0 331.0,308.0" fill="#7B5EA7"/><line x1="600.0" y1="302.0" x2="891.0" y2="302.0" stroke="#7B5EA7" stroke-width="2.6"/><polygon points="900.0,302.0 891.0,296.0 891.0,308.0" fill="#7B5EA7"/><text x="72.0" y="288.0" font-size="14" fill="#7B5EA7" text-anchor="start" font-weight="bold">h<tspan font-size="10.1" dy="3">t-1</tspan></text><text x="840.0" y="288.0" font-size="14" fill="#7B5EA7" text-anchor="start" font-weight="bold">h<tspan font-size="10.1" dy="3">t</tspan></text><text x="190.0" y="332.0" font-size="12.5" fill="#7B5EA7" text-anchor="middle" font-weight="normal">이전 Hidden State</text><text x="745.0" y="332.0" font-size="12.5" fill="#7B5EA7" text-anchor="middle" font-weight="normal">새 Hidden State</text><text x="902.0" y="240.0" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="normal">다음</text><text x="902.0" y="258.0" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="normal">시점</text><rect x="340.0" y="520.0" width="260.0" height="48.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="470.0" y="549.2" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">현재 단어 토큰</text><line x1="470.0" y1="520.0" x2="470.0" y2="502.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="470.0,494.0 464.0,502.0 476.0,502.0" fill="#8A93A0"/><rect x="340.0" y="446.0" width="260.0" height="48.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="470.0" y="475.2" font-size="14.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">Embedding</text><line x1="470.0" y1="446.0" x2="470.0" y2="353.0" stroke="#2E9E96" stroke-width="2.4"/><polygon points="470.0,344.0 464.0,353.0 476.0,353.0" fill="#2E9E96"/><text x="500.0" y="400.0" font-size="14" fill="#0E4A46" text-anchor="start" font-weight="bold">x<tspan font-size="10.1" dy="3">t</tspan></text><text x="470.0" y="400.0" font-size="12.5" fill="#0E4A46" text-anchor="end" font-weight="normal">현재 입력</text><rect x="40.0" y="380.0" width="250.0" height="96.0" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="60.0" y="406.0" font-size="13.5" fill="#555C68" text-anchor="start" font-weight="bold">선 구분</text><line x1="62.0" y1="428.0" x2="106.0" y2="428.0" stroke="#7B5EA7" stroke-width="2.6"/><text x="116.0" y="433.0" font-size="12.5" fill="#3C4552" text-anchor="start" font-weight="normal">h  Hidden State</text><line x1="62.0" y1="452.0" x2="106.0" y2="452.0" stroke="#DE8A3E" stroke-width="2.2"/><line x1="62.0" y1="458.0" x2="106.0" y2="458.0" stroke="#DE8A3E" stroke-width="2.2"/><text x="116.0" y="460.0" font-size="12.5" fill="#3C4552" text-anchor="start" font-weight="normal">c  Cell State</text><text x="470.0" y="604.0" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="bold">SimpleRNN 은 h 하나만, LSTM 은 h 와 c 두 개를 다음 시점으로 전달합니다.</text></svg>


### 그림에서 확인할 점
- 현재 단어 토큰은 **Embedding 을 거쳐 입력 벡터 `xₜ`** 가 되어 LSTM 에 들어갑니다.
- LSTM 은 이전 `h` 와 이전 `c` 를 **함께** 받습니다.
- 그리고 새 `h` 와 새 `c` **두 개**를 다음 시점으로 내보냅니다.
- 보라색 실선이 `h`, 주황색 이중선이 `c` 입니다.

---
## 08-3. Gate 세 개

> **Gate** — 어떤 정보를 얼마나 남기고, 넣고, 내보낼지 조절하는 장치입니다.

| 이름 | 하는 일 |
|---|---|
| **Forget Gate** | 이전 Cell State 에서 **무엇을 얼마나 남길지** 조절 |
| **Input Gate** | 현재 입력에서 얻은 **새 정보를 얼마나 반영할지** 조절 |
| **Output Gate** | 현재 상태 중 무엇을 **Hidden State 로 얼마나 내보낼지** 조절 |


<svg viewBox="0 0 940 620" width="100%" role="img" aria-label="LSTM의 Forget Input Output 세 Gate가 조절하는 대상과 판단 근거" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>LSTM 의 세 Gate</title><desc>Cell State 통로 아래에 Forget Gate Input Gate Output Gate 세 상자를 놓고 각각 0에서 1 사이의 반영 비율 막대를 그려, Gate가 켜고 끄는 스위치가 아니라 정도를 조절하는 장치임을 보여 주고, 세 Gate가 모두 현재 입력과 이전 Hidden State를 함께 보고 판단한다는 점을 아래쪽에서 점선 화살표로 연결한 그림</desc><rect x="0" y="0" width="940" height="620" fill="#FFFFFF"/><text x="470.0" y="34.0" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">Gate 는 정보를 “얼마나” 반영할지 조절합니다</text><text x="470.0" y="58.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">켜고 끄는 스위치가 아니라 0 과 1 사이의 비율입니다</text><line x1="30.0" y1="113.0" x2="890.0" y2="113.0" stroke="#DE8A3E" stroke-width="2.2"/><line x1="30.0" y1="119.0" x2="890.0" y2="119.0" stroke="#DE8A3E" stroke-width="2.2"/><line x1="880.0" y1="116.0" x2="896.0" y2="116.0" stroke="#DE8A3E" stroke-width="2.2"/><polygon points="906.0,116.0 896.0,109.0 896.0,123.0" fill="#DE8A3E"/><text x="48.0" y="100.0" font-size="13.5" fill="#DE8A3E" text-anchor="start" font-weight="bold">c<tspan font-size="9.7" dy="3">t-1</tspan></text><text x="858.0" y="100.0" font-size="13.5" fill="#DE8A3E" text-anchor="start" font-weight="bold">c<tspan font-size="9.7" dy="3">t</tspan></text><text x="470.0" y="100.0" font-size="13" fill="#DE8A3E" text-anchor="middle" font-weight="bold">Cell State 통로</text><rect x="60.0" y="176.0" width="230.0" height="176.0" rx="10" ry="10" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><circle cx="86.0" cy="202.0" r="14.0" fill="#DE8A3E"/><text x="86.0" y="207.0" font-size="13" fill="#FFFFFF" text-anchor="middle" font-weight="bold">①</text><text x="190.0" y="202.0" font-size="16" fill="#77400F" text-anchor="middle" font-weight="bold">줄이기</text><text x="190.0" y="226.0" font-size="12.5" fill="#77400F" text-anchor="middle" font-weight="normal" opacity="0.85">Forget Gate</text><rect x="102.0" y="244.0" width="146.0" height="11.0" rx="6" ry="6" fill="#FFFFFF" stroke="#DE8A3E" stroke-width="1.2"/><rect x="102.0" y="244.0" width="90.5" height="11.0" rx="6" ry="6" fill="#DE8A3E"/><text x="102.0" y="272.0" font-size="11.5" fill="#6B7280" text-anchor="start" font-weight="normal">0</text><text x="248.0" y="272.0" font-size="11.5" fill="#6B7280" text-anchor="end" font-weight="normal">1</text><text x="175.0" y="272.0" font-size="11.5" fill="#6B7280" text-anchor="middle" font-weight="normal">반영 비율</text><text x="175.0" y="300.0" font-size="11.5" fill="#77400F" text-anchor="middle" font-weight="normal">이전 Cell State 에서</text><text x="175.0" y="320.0" font-size="11.5" fill="#77400F" text-anchor="middle" font-weight="normal">무엇을 얼마나 남길지</text><line x1="175.0" y1="176.0" x2="175.0" y2="138.0" stroke="#DE8A3E" stroke-width="2.2"/><polygon points="175.0,130.0 169.0,138.0 181.0,138.0" fill="#DE8A3E"/><rect x="355.0" y="176.0" width="230.0" height="176.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><circle cx="381.0" cy="202.0" r="14.0" fill="#2E9E96"/><text x="381.0" y="207.0" font-size="13" fill="#FFFFFF" text-anchor="middle" font-weight="bold">②</text><text x="485.0" y="202.0" font-size="16" fill="#0E4A46" text-anchor="middle" font-weight="bold">새로 넣기</text><text x="485.0" y="226.0" font-size="12.5" fill="#0E4A46" text-anchor="middle" font-weight="normal" opacity="0.85">Input Gate</text><rect x="397.0" y="244.0" width="146.0" height="11.0" rx="6" ry="6" fill="#FFFFFF" stroke="#2E9E96" stroke-width="1.2"/><rect x="397.0" y="244.0" width="80.3" height="11.0" rx="6" ry="6" fill="#2E9E96"/><text x="397.0" y="272.0" font-size="11.5" fill="#6B7280" text-anchor="start" font-weight="normal">0</text><text x="543.0" y="272.0" font-size="11.5" fill="#6B7280" text-anchor="end" font-weight="normal">1</text><text x="470.0" y="272.0" font-size="11.5" fill="#6B7280" text-anchor="middle" font-weight="normal">반영 비율</text><text x="470.0" y="300.0" font-size="11.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">현재 입력에서 얻은 새 정보를</text><text x="470.0" y="320.0" font-size="11.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">얼마나 반영할지</text><line x1="470.0" y1="176.0" x2="470.0" y2="138.0" stroke="#2E9E96" stroke-width="2.2"/><polygon points="470.0,130.0 464.0,138.0 476.0,138.0" fill="#2E9E96"/><rect x="650.0" y="176.0" width="230.0" height="176.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><circle cx="676.0" cy="202.0" r="14.0" fill="#4E9A57"/><text x="676.0" y="207.0" font-size="13" fill="#FFFFFF" text-anchor="middle" font-weight="bold">③</text><text x="780.0" y="202.0" font-size="16" fill="#1D4726" text-anchor="middle" font-weight="bold">내보내기</text><text x="780.0" y="226.0" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal" opacity="0.85">Output Gate</text><rect x="692.0" y="244.0" width="146.0" height="11.0" rx="6" ry="6" fill="#FFFFFF" stroke="#4E9A57" stroke-width="1.2"/><rect x="692.0" y="244.0" width="70.1" height="11.0" rx="6" ry="6" fill="#4E9A57"/><text x="692.0" y="272.0" font-size="11.5" fill="#6B7280" text-anchor="start" font-weight="normal">0</text><text x="838.0" y="272.0" font-size="11.5" fill="#6B7280" text-anchor="end" font-weight="normal">1</text><text x="765.0" y="272.0" font-size="11.5" fill="#6B7280" text-anchor="middle" font-weight="normal">반영 비율</text><text x="765.0" y="300.0" font-size="11.5" fill="#1D4726" text-anchor="middle" font-weight="normal">현재 상태 중 무엇을</text><text x="765.0" y="320.0" font-size="11.5" fill="#1D4726" text-anchor="middle" font-weight="normal">h 로 얼마나 내보낼지</text><line x1="880.0" y1="264.0" x2="897.0" y2="264.0" stroke="#7B5EA7" stroke-width="2.4"/><polygon points="906.0,264.0 897.0,258.0 897.0,270.0" fill="#7B5EA7"/><text x="884.0" y="248.0" font-size="13.5" fill="#7B5EA7" text-anchor="start" font-weight="bold">h<tspan font-size="9.7" dy="3">t</tspan></text><text x="470.0" y="388.0" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="bold">세 Gate 는 모두 아래 두 가지를 함께 보고 판단합니다</text><line x1="175.0" y1="400.0" x2="175.0" y2="366.0" stroke="#CBD0D8" stroke-width="2.0" stroke-dasharray="5 4"/><polygon points="175.0,358.0 170.0,366.0 180.0,366.0" fill="#CBD0D8"/><line x1="470.0" y1="400.0" x2="470.0" y2="366.0" stroke="#CBD0D8" stroke-width="2.0" stroke-dasharray="5 4"/><polygon points="470.0,358.0 465.0,366.0 475.0,366.0" fill="#CBD0D8"/><line x1="765.0" y1="400.0" x2="765.0" y2="366.0" stroke="#CBD0D8" stroke-width="2.0" stroke-dasharray="5 4"/><polygon points="765.0,358.0 760.0,366.0 770.0,366.0" fill="#CBD0D8"/><rect x="150.0" y="404.0" width="640.0" height="104.0" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><rect x="180.0" y="428.0" width="260.0" height="56.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="310.0" y="453.0" font-size="14" fill="#1B3A5E" text-anchor="middle" font-weight="bold">현재 입력</text><text x="310.0" y="473.0" font-size="11.9" fill="#1B3A5E" text-anchor="middle" font-weight="normal" opacity="0.85">Embedding 을 거친 이번 토큰</text><text x="470.0" y="462.0" font-size="20" fill="#6B7280" text-anchor="middle" font-weight="normal">＋</text><rect x="500.0" y="428.0" width="260.0" height="56.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="630.0" y="453.0" font-size="14" fill="#3B295D" text-anchor="middle" font-weight="bold">이전 Hidden State</text><text x="630.0" y="473.0" font-size="11.9" fill="#3B295D" text-anchor="middle" font-weight="normal" opacity="0.85">직전 시점까지의 요약 상태</text><text x="470.0" y="546.0" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="bold">Gate 는 0 또는 1 만 고르는 스위치가 아닙니다.</text><text x="470.0" y="570.0" font-size="13.0" fill="#6B7280" text-anchor="middle" font-weight="normal">각 정보를 어느 정도 반영할지 연속적인 값으로 조절합니다. 이 노트북에서는 Gate 를 직접 구현하지 않습니다.</text></svg>


### 반드시 오해하지 말아야 할 점

> ### Gate 는 0 또는 1 만 고르는 ON / OFF 스위치가 아닙니다.
> ### 각 정보를 어느 정도 반영할지 **연속적인 값으로 조절**합니다.

### 세 Gate 는 무엇을 보고 판단하나요?

세 Gate 모두 **현재 입력 `xₜ`** 와 **이전 Hidden State `hₜ₋₁`** 를 함께 보고 판단합니다.
그림 아래쪽의 점선 화살표가 그 부분입니다.

> 각 Gate 의 계산식은 이번 노트북에서 다루지 않습니다.
> `tf.keras.layers.LSTM` 이 내부에서 알아서 계산합니다.

---
## 08-4. 코드에서 실제로 바뀌는 한 줄

| | SimpleRNN 모델 | LSTM 모델 |
|---|---|---|
| 1층 | `Embedding(VOCAB_SIZE, 32, mask_zero=True)` | `Embedding(VOCAB_SIZE, 32, mask_zero=True)` |
| 2층 | **`SimpleRNN(64)`** | **`LSTM(64)`** |
| 3층 | `Dense(VOCAB_SIZE, activation="softmax")` | `Dense(VOCAB_SIZE, activation="softmax")` |
| 학습 데이터 | `X`, `y` | `X`, `y` (같음) |
| 반복 횟수 | `EPOCHS` | `EPOCHS` (같음) |

> **Embedding, Dense, 학습 데이터는 동일하고 순환층 한 줄이 바뀝니다.**

In [11]:
lstm_model = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(MAX_LEN,)),

    # 단어 번호 → 숫자 벡터  (SimpleRNN 모델과 완전히 동일)
    tf.keras.layers.Embedding(
        VOCAB_SIZE,
        32,
        mask_zero=True,
    ),

    # ==================================================
    # ★ LSTM 핵심
    #
    # SimpleRNN 과 마찬가지로 단어를 순서대로 읽지만,
    # Hidden State(h) 외에 Cell State(c) 를 사용합니다.
    #
    # Gate 들이
    #   - 무엇을 줄일지
    #   - 무엇을 새로 저장할지
    #   - 무엇을 출력할지
    # 조절하여 필요한 정보를 더 오래 전달하도록 설계되어 있습니다.
    # ==================================================
    tf.keras.layers.LSTM(64),

    # 다음 단어 후보 점수  (SimpleRNN 모델과 완전히 동일)
    tf.keras.layers.Dense(
        VOCAB_SIZE,
        activation="softmax",
    ),
])

---
# 09. LSTM 학습 및 같은 입력 비교

## 09-1. 같은 조건으로 학습시키기

In [12]:
# ============================================================
# [핵심 코드] LSTM 학습 — SimpleRNN 과 완전히 같은 조건
#
#  입력  : X, y  (SimpleRNN 이 쓴 것과 같은 데이터)
#  하는 일: 순환층만 LSTM 인 모델을 같은 EPOCHS 만큼 학습합니다.
#  출력  : 학습이 끝난 lstm_model 과 Loss 기록
#  다음  : 09-2 에서 두 모델에 같은 입력을 넣어 비교합니다.
# ============================================================
lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
)

lstm_history = lstm_model.fit(
    X, y,                    # ← SimpleRNN 과 완전히 같은 학습 데이터
    epochs=EPOCHS,           # ← 같은 반복 횟수
    validation_split=0.2,
    verbose=0,
)

print(
    f"LSTM Loss: "
    f"{lstm_history.history['loss'][0]:.3f}"
    f" → {lstm_history.history['loss'][-1]:.3f}"
)

LSTM Loss: 4.316 → 0.083


---
## 09-2. 같은 입력을 두 모델에 넣어 보기

두 모델은 다음이 모두 같습니다.

- 같은 문장(corpus) · 같은 단어 사전 · 같은 학습 데이터 `X` / `y`
- 같은 Embedding 크기(32) · 같은 hidden 크기(64) · 같은 epoch

**다른 것은 순환층 한 줄뿐입니다.**

### 세 번째 입력을 눈여겨보세요

세 번째는 **긴 문맥** 예제입니다.
바로 앞 토큰은 `담당자에게` 로 짧은 예제와 같지만, 마지막 단어를 결정하는 것은
**문장 맨 앞의 도입 부분**입니다.

> **미리 알아 둘 점 — LSTM 이 항상 이기는 것은 아닙니다**
>
> LSTM 은 “언제나 더 정확한 모델”이 아닙니다.
> **긴 의존 관계를 다루기 쉽도록 Cell State 와 Gate 구조가 추가된 모델**입니다.
> 작은 교육용 데이터에서는 SimpleRNN 이 더 잘 맞는 예도 있을 수 있습니다.
> 아래 결과도 조작하지 않고 실행된 그대로 보여 줍니다.

In [13]:
# ============================================================
# [결과 확인용 보조 코드 — RNN/LSTM 핵심 알고리즘이 아닙니다]
#
# 아래 코드는 예측 결과를 보기 쉽게 출력하기 위한 코드입니다.
# Python 문법을 한 줄씩 분석할 필요는 없습니다.
# ============================================================
비교문장 = [
    "자료를 확인하고 담당자에게",
    "회의 자료를 검토한 후 결과를",
    "보완 요청을 받은 후 관련 자료를 검토하고 필요한 내용을 정리하여 담당자에게",   # 긴 문맥
]

for 문장 in 비교문장:
    r1 = 다음단어_후보(rnn_model, 문장, k=1)[0]
    l1 = 다음단어_후보(lstm_model, 문장, k=1)[0]
    print("=" * 70)
    print(f'입력 : "{문장} ___"')
    print(f"  SimpleRNN 1순위 : {r1[0]}  ({r1[1]:.3f})")
    print(f"  LSTM      1순위 : {l1[0]}  ({l1[1]:.3f})")
    print()

입력 : "자료를 확인하고 담당자에게 ___"
  SimpleRNN 1순위 : 전달했습니다  (0.996)
  LSTM      1순위 : 전달했습니다  (0.990)

입력 : "회의 자료를 검토한 후 결과를 ___"
  SimpleRNN 1순위 : 공유했습니다  (0.994)
  LSTM      1순위 : 공유했습니다  (0.976)

입력 : "보완 요청을 받은 후 관련 자료를 검토하고 필요한 내용을 정리하여 담당자에게 ___"
  SimpleRNN 1순위 : 결과를  (0.866)
  LSTM      1순위 : 제출했습니다  (0.961)



---
# 10. Attention 으로 연결

RNN 과 LSTM 은 앞에서부터 상태를 계속 전달합니다.
문장이 길어질수록 **필요한 정보를 하나의 상태 흐름에 계속 담아 전달**해야 합니다.

다음 실습에서 배울 **Attention** 은 방식이 다릅니다.
상태 하나에 모두 담는 대신, **필요한 위치를 직접 다시 참고**합니다.


<svg width="100%" viewBox="0 0 940 600" role="img" aria-label="RNN LSTM 의 순차 누적 방식과 다음 과정인 Attention 의 차이" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>RNN LSTM 의 순차 누적 방식과 다음 과정인 Attention 의 차이</title><desc>왼쪽은 RNN과 LSTM이 단어를 순서대로 읽어 하나의 상태에 정보를 누적하는 방식이고, 오른쪽은 다음 과정인 Attention이 예측하는 시점에서 앞쪽 단어들을 선택적으로 더 참고하는 방식임을 나란히 비교한 그림</desc><rect x="0" y="0" width="940" height="600" fill="#FFFFFF"/><text x="470.0" y="34" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">다음 단어를 예측할 때, 앞의 모든 단어가 똑같이 중요할까요?</text><text x="470.0" y="58" font-size="13.5" fill="#6B7280" text-anchor="middle" font-weight="normal">“지난 회의에서 정한 기준에 따라 여러 자료를 검토한 뒤 담당자에게 ______”</text><rect x="28" y="84" width="420" height="400" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="238" y="114" font-size="17" fill="#3B295D" text-anchor="middle" font-weight="bold">지금까지 배운 방법</text><text x="238" y="136" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">RNN / LSTM</text><rect x="58" y="158" width="360" height="50" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="238" y="188" font-size="14" fill="#3B295D" text-anchor="middle" font-weight="bold">단어를 순서대로 하나씩 읽습니다</text><line x1="238" y1="208" x2="238" y2="223" stroke="#8A93A0" stroke-width="2"/><polygon points="238,232 232,222 244,222" fill="#8A93A0"/><rect x="58" y="232" width="360" height="62" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="238" y="258" font-size="14" fill="#3B295D" text-anchor="middle" font-weight="bold">이전 상태를 다음 시점으로 전달합니다</text><text x="238" y="278" font-size="12.5" fill="#3B295D" text-anchor="middle" font-weight="normal" opacity="0.8">Hidden State · Cell State</text><line x1="238" y1="294" x2="238" y2="309" stroke="#8A93A0" stroke-width="2"/><polygon points="238,318 232,308 244,308" fill="#8A93A0"/><rect x="58" y="318" width="360" height="50" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="238" y="348" font-size="14" fill="#3B295D" text-anchor="middle" font-weight="bold">지금까지의 정보를 상태 하나에 누적합니다</text><line x1="238" y1="368" x2="238" y2="383" stroke="#8A93A0" stroke-width="2"/><polygon points="238,392 232,382 244,382" fill="#8A93A0"/><rect x="58" y="384" width="360" height="50" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="238" y="414" font-size="14" fill="#1D4726" text-anchor="middle" font-weight="bold">마지막 상태로 다음 단어 후보 계산</text><text x="238" y="462" font-size="12.5" fill="#77400F" text-anchor="middle" font-weight="normal">앞쪽 정보일수록 여러 시점을 지나며 영향이 약해질 수 있습니다</text><rect x="492" y="84" width="420" height="400" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="702" y="114" font-size="17" fill="#0E4A46" text-anchor="middle" font-weight="bold">다음 과정에서 배울 방법</text><text x="702" y="136" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">Attention</text><text x="702" y="156" font-size="11" fill="#6B7280" text-anchor="middle" font-weight="normal">※ 비율 숫자는 개념 설명용 예시이며 이 노트북의 실행 결과가 아닙니다</text><rect x="516" y="172" width="218" height="38" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="608" y="196" font-size="13" fill="#0E4A46" text-anchor="middle" font-weight="bold">지난 회의에서 정한 기준</text><text x="726" y="196" font-size="11.5" fill="#2E9E96" text-anchor="end" font-weight="normal">95%</text><line x1="740" y1="191" x2="805" y2="376" stroke="#2E9E96" stroke-width="4.8" opacity="0.82"/><rect x="516" y="224" width="218" height="38" rx="8" ry="8" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="608" y="248" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">여러 자료를</text><text x="726" y="248" font-size="11.5" fill="#6B7280" text-anchor="end" font-weight="normal">35%</text><line x1="740" y1="243" x2="805" y2="376" stroke="#2E9E96" stroke-width="2.4" opacity="0.46"/><rect x="516" y="276" width="218" height="38" rx="8" ry="8" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="608" y="300" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">검토한 뒤</text><text x="726" y="300" font-size="11.5" fill="#6B7280" text-anchor="end" font-weight="normal">25%</text><line x1="740" y1="295" x2="805" y2="376" stroke="#2E9E96" stroke-width="2.0" opacity="0.40"/><rect x="516" y="328" width="218" height="38" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="608" y="352" font-size="13" fill="#0E4A46" text-anchor="middle" font-weight="bold">담당자에게</text><text x="726" y="352" font-size="11.5" fill="#2E9E96" text-anchor="end" font-weight="normal">80%</text><line x1="740" y1="347" x2="805" y2="376" stroke="#2E9E96" stroke-width="4.2" opacity="0.73"/><rect x="758" y="358" width="132" height="44" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="824" y="386" font-size="14" fill="#1D4726" text-anchor="middle" font-weight="bold">다음 단어</text><text x="660" y="436" font-size="13" fill="#0E4A46" text-anchor="middle" font-weight="bold">예측하는 시점에서 필요한 앞쪽 정보를</text><text x="660" y="458" font-size="13" fill="#0E4A46" text-anchor="middle" font-weight="bold">선택적으로 더 많이 참고하는 방법입니다</text><rect x="96" y="508" width="748" height="68" rx="12" ry="12" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="470.0" y="536" font-size="14" fill="#77400F" text-anchor="middle" font-weight="bold">다음 Attention 강의에서는 어떤 위치의 정보를 더 참고할지 살펴보고,</text><text x="470.0" y="559" font-size="14" fill="#77400F" text-anchor="middle" font-weight="bold">이후 Transformer 에서는 Self-Attention 으로 위치 간 관계를 계산하는 구조로 이어집니다.</text></svg>


> Attention 의 계산 방법과 구현은 **이번 노트북에서 다루지 않습니다.**
> 여기서는 “왜 다음 단계가 필요한가”만 확인하고 넘어갑니다.

---
# 11. 핵심 정리

## 11-1. SimpleRNN 전체 흐름

| 순서 | 단계 | 이때 무슨 일이 일어나나 |
|---|---|---|
| 1 | 단어 토큰 | 문장을 띄어쓰기 기준으로 나눈 조각 |
| 2 | Token ID | 토큰마다 번호를 붙임 |
| 3 | Embedding | 번호 → 숫자 벡터 `x` |
| 4 | **SimpleRNN** | `x` 와 **이전 h** 를 함께 받아 **새 h** 를 만듦 |
| 5 | **상태 전달** | 새 h 를 다음 시점으로 넘김 (시점마다 반복) |
| 6 | 마지막 h | 문장 앞부분 전체를 처리한 결과 |
| 7 | Dense + Softmax | 모든 단어 토큰에 점수를 매김 |
| 8 | 다음 단어 후보 | 점수가 높은 순서대로 제시 |

---
## 11-2. LSTM 전체 흐름

| 순서 | 단계 | 이때 무슨 일이 일어나나 |
|---|---|---|
| 1 | 단어 토큰 | SimpleRNN 과 동일 |
| 2 | Token ID | SimpleRNN 과 동일 |
| 3 | Embedding | SimpleRNN 과 동일 |
| 4 | **LSTM** | `x` 와 **이전 h**, **이전 c** 를 함께 받음 |
| 5 | **Gate 조절** | Forget · Input · Output 이 정보량을 조절 |
| 6 | **상태 전달** | **새 h 와 새 c 두 개**를 다음 시점으로 넘김 |
| 7 | 마지막 h | 문장 앞부분 전체를 처리한 결과 |
| 8 | Dense + Softmax | SimpleRNN 과 동일 |
| 9 | 다음 단어 후보 | SimpleRNN 과 동일 |

---
## 11-3. SimpleRNN vs LSTM


<svg viewBox="0 0 940 720" width="100%" role="img" aria-label="SimpleRNN과 LSTM의 전체 흐름을 나란히 놓은 최종 비교" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>SimpleRNN 과 LSTM 최종 비교</title><desc>왼쪽에 SimpleRNN, 오른쪽에 LSTM의 흐름을 단어 토큰부터 Embedding 순환층 상태 전달 마지막 Hidden State Dense 다음 단어 후보까지 같은 순서로 나란히 배치하여, 순환층 한 줄과 전달하는 상태의 개수만 다르고 나머지는 모두 같음을 보여 주는 그림</desc><rect x="0" y="0" width="940" height="720" fill="#FFFFFF"/><text x="470.0" y="34.0" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">전체 구조는 거의 같고, 바뀌는 것은 순환층 한 줄입니다</text><text x="470.0" y="58.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">왼쪽 SimpleRNN · 오른쪽 LSTM</text><rect x="60.0" y="80.0" width="380.0" height="552.0" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="250.0" y="108.0" font-size="17" fill="#1F2733" text-anchor="middle" font-weight="bold">SimpleRNN</text><rect x="500.0" y="80.0" width="380.0" height="552.0" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="690.0" y="108.0" font-size="17" fill="#1F2733" text-anchor="middle" font-weight="bold">LSTM</text><rect x="110.0" y="124.0" width="280.0" height="44.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="250.0" y="150.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">단어 토큰</text><line x1="250.0" y1="168.0" x2="250.0" y2="176.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,184.0 255.0,176.0 245.0,176.0" fill="#8A93A0"/><rect x="550.0" y="124.0" width="280.0" height="44.0" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="690.0" y="150.9" font-size="13.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">단어 토큰</text><line x1="690.0" y1="168.0" x2="690.0" y2="176.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,184.0 695.0,176.0 685.0,176.0" fill="#8A93A0"/><rect x="110.0" y="186.0" width="280.0" height="44.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="250.0" y="212.9" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">Embedding</text><line x1="250.0" y1="230.0" x2="250.0" y2="238.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,246.0 255.0,238.0 245.0,238.0" fill="#8A93A0"/><rect x="550.0" y="186.0" width="280.0" height="44.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="690.0" y="212.9" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">Embedding</text><line x1="690.0" y1="230.0" x2="690.0" y2="238.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,246.0 695.0,238.0 685.0,238.0" fill="#8A93A0"/><rect x="110.0" y="248.0" width="280.0" height="44.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="2.6"/><text x="250.0" y="274.9" font-size="13.5" fill="#3B295D" text-anchor="middle" font-weight="normal">SimpleRNN(64)</text><text x="94.0" y="276.0" font-size="15" fill="#DE8A3E" text-anchor="middle" font-weight="normal">★</text><line x1="250.0" y1="292.0" x2="250.0" y2="300.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,308.0 255.0,300.0 245.0,300.0" fill="#8A93A0"/><rect x="550.0" y="248.0" width="280.0" height="44.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="2.6"/><text x="690.0" y="274.9" font-size="13.5" fill="#3B295D" text-anchor="middle" font-weight="normal">LSTM(64)</text><text x="534.0" y="276.0" font-size="15" fill="#DE8A3E" text-anchor="middle" font-weight="normal">★</text><line x1="690.0" y1="292.0" x2="690.0" y2="300.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,308.0 695.0,300.0 685.0,300.0" fill="#8A93A0"/><rect x="110.0" y="310.0" width="280.0" height="44.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="2.6"/><text x="250.0" y="336.9" font-size="13.5" fill="#3B295D" text-anchor="middle" font-weight="normal">h 를 다음 시점으로 전달</text><text x="94.0" y="338.0" font-size="15" fill="#DE8A3E" text-anchor="middle" font-weight="normal">★</text><line x1="250.0" y1="354.0" x2="250.0" y2="362.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,370.0 255.0,362.0 245.0,362.0" fill="#8A93A0"/><rect x="550.0" y="310.0" width="280.0" height="44.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="2.6"/><text x="690.0" y="336.9" font-size="13.5" fill="#3B295D" text-anchor="middle" font-weight="normal">h 와 c 를 다음 시점으로 전달</text><text x="534.0" y="338.0" font-size="15" fill="#DE8A3E" text-anchor="middle" font-weight="normal">★</text><line x1="690.0" y1="354.0" x2="690.0" y2="362.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,370.0 695.0,362.0 685.0,362.0" fill="#8A93A0"/><rect x="110.0" y="372.0" width="280.0" height="44.0" rx="10" ry="10" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="250.0" y="398.9" font-size="13.5" fill="#555C68" text-anchor="middle" font-weight="normal">… 시점 반복 …</text><line x1="250.0" y1="416.0" x2="250.0" y2="424.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,432.0 255.0,424.0 245.0,424.0" fill="#8A93A0"/><rect x="550.0" y="372.0" width="280.0" height="44.0" rx="10" ry="10" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="690.0" y="398.9" font-size="13.5" fill="#555C68" text-anchor="middle" font-weight="normal">… 시점 반복 …</text><line x1="690.0" y1="416.0" x2="690.0" y2="424.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,432.0 695.0,424.0 685.0,424.0" fill="#8A93A0"/><rect x="110.0" y="434.0" width="280.0" height="44.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="250.0" y="460.9" font-size="13.5" fill="#3B295D" text-anchor="middle" font-weight="normal">마지막 h</text><line x1="250.0" y1="478.0" x2="250.0" y2="486.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,494.0 255.0,486.0 245.0,486.0" fill="#8A93A0"/><rect x="550.0" y="434.0" width="280.0" height="44.0" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="690.0" y="460.9" font-size="13.5" fill="#3B295D" text-anchor="middle" font-weight="normal">마지막 h</text><line x1="690.0" y1="478.0" x2="690.0" y2="486.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,494.0 695.0,486.0 685.0,486.0" fill="#8A93A0"/><rect x="110.0" y="496.0" width="280.0" height="44.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="250.0" y="522.9" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">Dense + Softmax</text><line x1="250.0" y1="540.0" x2="250.0" y2="548.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="250.0,556.0 255.0,548.0 245.0,548.0" fill="#8A93A0"/><rect x="550.0" y="496.0" width="280.0" height="44.0" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="690.0" y="522.9" font-size="13.5" fill="#0E4A46" text-anchor="middle" font-weight="normal">Dense + Softmax</text><line x1="690.0" y1="540.0" x2="690.0" y2="548.0" stroke="#8A93A0" stroke-width="2.0"/><polygon points="690.0,556.0 695.0,548.0 685.0,548.0" fill="#8A93A0"/><rect x="110.0" y="558.0" width="280.0" height="44.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="250.0" y="584.9" font-size="13.5" fill="#1D4726" text-anchor="middle" font-weight="normal">다음 단어 후보</text><rect x="550.0" y="558.0" width="280.0" height="44.0" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="690.0" y="584.9" font-size="13.5" fill="#1D4726" text-anchor="middle" font-weight="normal">다음 단어 후보</text><rect x="60.0" y="648.0" width="380.0" height="54.0" rx="10" ry="10" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="250.0" y="672.0" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="bold">공통</text><text x="250.0" y="692.0" font-size="13" fill="#3C4552" text-anchor="middle" font-weight="normal">Embedding · Dense · 학습 데이터</text><rect x="500.0" y="648.0" width="380.0" height="54.0" rx="10" ry="10" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="690.0" y="672.0" font-size="13" fill="#77400F" text-anchor="middle" font-weight="bold">★ 핵심 차이</text><text x="690.0" y="692.0" font-size="13" fill="#77400F" text-anchor="middle" font-weight="normal">SimpleRNN(64) ↔ LSTM(64) · h ↔ h + c</text></svg>


### 코드로 보면 차이는 한 줄입니다

| SimpleRNN | LSTM |
|---|---|
| `tf.keras.layers.SimpleRNN(64)` | `tf.keras.layers.LSTM(64)` |
| `h` 만 전달 | `h` 와 `c` 를 Gate 로 조절하며 전달 |

---
## 11-4. 자기 점검

이 노트북을 끝까지 본 뒤, 다음 질문에 **자신의 말로** 답할 수 있으면 충분합니다.

| # | 질문 | 확인할 곳 |
|---|---|---|
| 1 | 왜 단어 순서가 중요한가? | 01-1 |
| 2 | 이번 실습에서는 문장을 어떤 단위로 나누는가? | 03-1 |
| 3 | Embedding 은 무엇을 하는가? | 03-3 |
| 4 | RNN 에 실제로 들어가는 것은 단어 문자열 자체인가? | 03-3 · 05-2 |
| 5 | Hidden State 는 무엇인가? | 05-1 |
| 6 | Hidden State 에 단어가 그대로 저장되는가? | 05-1 |
| 7 | 이전 Hidden State 는 다음 시점에 어떻게 전달되는가? | 05-2 |
| 8 | 같은 RNN 이 각 시점에서 반복 사용된다는 것은 무슨 뜻인가? | 05-2 |
| 9 | 마지막 Hidden State 는 어떻게 다음 단어 후보로 연결되는가? | 05-3 |
| 10 | 단어 순서를 바꾸면 왜 h 의 경로가 달라지는가? | 07-1 · 07-2 |
| 11 | SimpleRNN 은 긴 문맥에서 왜 어려울 수 있는가? | 08-1 |
| 12 | LSTM 은 SimpleRNN 에 무엇을 추가했는가? | 08-2 |
| 13 | Cell State 는 무엇인가? | 08-2 |
| 14 | Forget / Input / Output Gate 는 각각 어떤 역할인가? | 08-3 |
| 15 | Gate 는 단순한 ON / OFF 스위치인가? | 08-3 |
| 16 | 코드에서 SimpleRNN 과 LSTM 의 핵심 차이는 어디인가? | 08-4 · 11-3 |
| 17 | LSTM 이 항상 SimpleRNN 보다 정확한가? | 09-2 |
| 18 | 왜 다음 단계에서 Attention 이 등장하는가? | 10 |

---

### 마지막으로 기억할 것

1. **핵심은 “순서대로 읽고, 이전 상태를 다음 시점에 전달한다”는 것입니다.**
   Hidden State 는 지금까지 처리한 입력의 영향을 요약한 숫자 벡터이며, 단어가 그대로 저장된 것이 아닙니다.
2. **LSTM 은 Cell State 와 Gate 로 정보의 반영 정도를 조절합니다.**
   Gate 는 켜고 끄는 스위치가 아니라 0 과 1 사이의 비율입니다.
3. **LSTM 이 언제나 SimpleRNN 보다 좋은 결과를 내는 것은 아닙니다.**
   긴 의존 관계를 다루기 쉽도록 구조가 **추가**되었다는 것이 핵심입니다.
4. **모델의 출력은 정답 선언이 아니라 후보 제안입니다.**
   같은 앞부분 뒤에 여러 표현이 자연스럽게 올 수 있습니다.
5. **이번 실습은 원리 이해용 교육 예제입니다.**
   문장이 28개뿐이므로 정확도 수치에는 의미를 두지 않습니다.
6. **최종 판단과 책임은 담당자에게 있습니다.**
   실제 업무에 적용할 때는 개인정보와 내부 비공개 자료를 입력하지 않습니다.

> 다음 시간에는 **Attention** 으로 이어집니다.